In [ ]:
import zipfile
from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/high_plains_quifer.zip'
extract_path = '/content/ogallala_shp'

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted to:", extract_path)

!pip install -q xee xarray netcdf4 geopandas pyproj dask h5netcdf

Mounted at /content/drive
Files extracted to: /content/ogallala_shp
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.3 MB/s eta 0:00:00


In [ ]:
import os, sys, glob, time, shutil, logging, gc, hashlib
from datetime import datetime, timezone

import numpy as np
import ee
import xarray as xr
import xee
from xee import helpers
import geopandas as gpd
import pyproj
import dask

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('gee_export')
logging.getLogger('urllib3.connectionpool').setLevel(logging.ERROR)
DASK_WORKERS = 4
dask.config.set(scheduler='threads', num_workers=DASK_WORKERS)
PROJECT_ID = "msugw-503806"
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
log.info('Earth Engine ready  (project=%s)', PROJECT_ID)



In [ ]:
DATASET_ID = "projects/sat-io/open-datasets/HRES-WTD"
print(ee.ImageCollection(DATASET_ID).getInfo())

{'type': 'ImageCollection', 'bands': [], 'version': 1777903668810979, 'id': 'projects/sat-io/open-datasets/HRES-WTD', 'features': [{'type': 'Image', 'bands': [{'id': 'b1', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [19753, 23242], 'crs': 'EPSG:4326', 'crs_transform': [0.0002275049559242481, 0, -88.790745787833, 0, -0.0002275049559242481, 35.25092309655129]}], 'version': 1777903667543402, 'id': 'projects/sat-io/open-datasets/HRES-WTD/wtd_alabama', 'properties': {'system:footprint': {'type': 'LinearRing', 'coordinates': [[-85.98208335098876, 29.962989865094634], [-85.63101480802594, 29.96308289809589], [-85.27994618834161, 29.962989797704665], [-84.8586638432239, 29.962989879502974], [-84.29669106414684, 29.962691279946117], [-84.29667508286471, 35.25103705251898], [-84.71823648682467, 35.25103721234569], [-85.5608010481788, 35.25103722220794], [-86.40336569069302, 35.25103723036694], [-87.10550285090102, 35.25103720889061], [-87.80764003296595, 35.2510372427

In [ ]:
import ee

# Make sure you are authenticated/initialized first
# ee.Initialize(project='msugw-503806')

dataset_id = "projects/sat-io/open-datasets/HRES-WTD"
coll = ee.ImageCollection(dataset_id)

# Get the number of images
n_images = coll.size().getInfo()
print(f"Total images in collection: {n_images}\n")

# Grab the first image to inspect its metadata
first_img = coll.first()

# 1. Print Band Names
band_names = first_img.bandNames().getInfo()
print("Band Names:", band_names)

# 2. Print Band Types (Precision, Min, Max)
band_types = first_img.bandTypes().getInfo()
print("\nBand Types Info:")
for b, info in band_types.items():
    print(f"  {b}: {info}")

# 3. Check Date (Crucial for your pipeline's year filtering)
dates = coll.aggregate_array('system:time_start').getInfo()
if dates:
    print("\nTimestamps (first 5):")
    for d in dates[:5]:
        print(f"  {ee.Date(d).format('YYYY-MM-dd').getInfo()}")
else:
    print("\nNo system:time_start timestamps found! (See warning below)")

Total images in collection: 49

Band Names: ['b1']

Band Types Info:
  b1: {'type': 'PixelType', 'precision': 'float'}

No system:time_start timestamps found! (See warning below)


# HiHydroSoil v2.0

### ee.Image('projects/sat-io/open-datasets/HiHydroSoilv2_0/Hydrologic_Soil_Group_250m');

In [ ]:
# ==========================================
# 1. Install Required Packages & Setup Logging
# ==========================================

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
gee_to_netcdf_v3.py — Earth Engine → CF-Compliant NetCDF Exporter
===================================================================

Changes from v2
---------------
FIX(16) scale_factor/add_offset written per-band as CF variable attributes
        so raw packed values can be decoded by any CF-compliant reader.
FIX(17) Per-band _FillValue: each variable gets its own native fill value.
        QA bitmask bands get uint16 dtype; angle bands get scale=0.01; etc.
FIX(18) grid_mapping variable (`spatial_ref`) added to every file.
        Contains full CRS WKT, GeoTransform, and CF parameters.
        Every data variable references it via grid_mapping='spatial_ref'.
FIX(19) Native integer dtypes preserved (int16 for NDVI, uint16 for QA).
        v2 cast everything to float32, losing packing + fill semantics.
FIX(20) Auto-detect band dtypes from GEE when BAND_METADATA is empty.
        Falls back to the detected dtype, logs warning about missing scale.

All v2 crash/logic/integrity fixes (FIX 1-15) are carried forward.
"""

#
# ║  CONFIGURATION                                                     ║
# ╚════════════════════════════════════════════════════════════════════╝

DATASET_ID = 'projects/sat-io/open-datasets/HiHydroSoilv2_0/Hydrologic_Soil_Group_250m'


SHAPEFILE_DIR = "/content/ogallala_shp"
ROI_LABEL     = "Ogallala"
DRIVE_ROOT    = "/content/drive/MyDrive/MSUGWB"

YEAR_START = None
YEAR_END   = None

DASK_WORKERS    = 4
CHUNK_XY        = 2048
COMPRESS_LEVEL  = 0
MAX_RETRIES     = 3
FORCE_OVERWRITE = False

# ── Per-Band Metadata (FIX 16/17/19) ────────────────────────────────
#
# Source: GEE Data Catalog for MODIS/061/MOD13A3
# https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13A3
#
# Keys per band:
#   dtype        – NetCDF storage type (int16, uint16, float32, …)
#   scale_factor – CF scale: physical = raw × scale + offset
#   add_offset   – CF offset (usually 0.0)
#   _FillValue   – raw value meaning "no data"
#   valid_range   – [min, max] of meaningful raw values (optional, CF attr)
#   long_name    – human-readable description (CF attr)
#   units        – physical units after decoding (CF attr)
#
# For a different dataset:
#   1. Change DATASET_ID
#   2. Replace BAND_METADATA with entries from the GEE catalog page
#   3. If you don't know the metadata, set BAND_METADATA = {}
#      → the pipeline will auto-detect dtypes and warn about missing scale
#

BAND_METADATA = {
    'b1': {
        'dtype':        'uint8',
        '_FillValue':   255,
        'valid_range':  [1, 4],
        'long_name':    'Hydrologic Soil Group (1=A, 2=B, 3=C, 4=D)',
        'units':        '1',
        'flag_values':  np.array([1, 2, 3, 4], dtype='uint8'),
        'flag_meanings':'A B C D',
    },
}


# ╔════════════════════════════════════════════════════════════════════╗
# ║  SETUP                                                             ║
# ╚════════════════════════════════════════════════════════════════════╝

# --- Colab prerequisites ---
# Uncomment the next two lines when running on Colab:
# !pip install -q "xee>=0.0.14" xarray netcdf4 geopandas pyproj
# from google.colab import drive; drive.mount('/content/drive')


# ╔════════════════════════════════════════════════════════════════════╗
# ║  EXCEPTIONS                                                        ║
# ╚════════════════════════════════════════════════════════════════════╝

class DataQualityError(Exception):
    """Non-retryable data quality error."""
    pass


# ╔════════════════════════════════════════════════════════════════════╗
# ║  PIPELINE FUNCTIONS                                                ║
# ╚════════════════════════════════════════════════════════════════════╝

# ─── ROI ─────────────────────────────────────────────────────────────

def load_roi(shapefile_dir, simplify_deg=0.001):
    '''Load shapefile -> dissolved EPSG:4326 geometry.'''
    paths = glob.glob(os.path.join(shapefile_dir, '**', '*.shp'), recursive=True)
    if not paths:
        raise FileNotFoundError(f'No .shp found in {shapefile_dir}')
    log.info('Shapefile: %s', paths[0])

    gdf = gpd.read_file(paths[0])
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    roi = (gdf.geometry.union_all()
           if hasattr(gdf.geometry, 'union_all')
           else gdf.geometry.unary_union)

    try:
        import shapely as _shp
        nv = int(_shp.get_num_coordinates(roi))
    except Exception:
        nv = len(roi.wkt) // 20

    if nv > 50_000:
        roi = roi.simplify(simplify_deg, preserve_topology=True)
        try:
            nv_after = int(_shp.get_num_coordinates(roi))
        except Exception:
            nv_after = len(roi.wkt) // 20
        log.warning('Simplified ROI: %s -> %s vertices (tol=%.4f deg)',
                    f'{nv:,}', f'{nv_after:,}', simplify_deg)
    else:
        log.info('ROI vertices: %s', f'{nv:,}')
    return roi


# ─── CRS & Grid ─────────────────────────────────────────────────────

def resolve_native_crs(collection):
    '''Detect native CRS + nominal scale.  Returns (epsg_str, metres).'''
    proj = collection.select(0).projection()
    info = proj.getInfo()
    native_crs = info.get('crs', 'EPSG:4326')
    nominal_m  = proj.nominalScale().getInfo()

    log.info('Native CRS    : %s', native_crs[:70])
    log.info('Nominal scale : %.1f m', nominal_m)

    resolved = 'EPSG:4326'

    if native_crs.startswith('SR-ORG:'):
        log.warning('SR-ORG CRS (%s) -> EPSG:4326 fallback', native_crs)
    elif native_crs.startswith('EPSG:'):
        resolved = native_crs
    else:
        try:
            crs_obj = pyproj.CRS.from_user_input(native_crs)
            epsg = crs_obj.to_epsg()
            if epsg:
                resolved = f'EPSG:{epsg}'
            elif crs_obj.axis_info and crs_obj.axis_info[0].unit_name == 'metre':
                resolved = 'EPSG:5070'
                log.warning('Metre CRS, no EPSG -> EPSG:5070')
        except Exception as exc:
            log.warning('CRS parse error: %s -> EPSG:4326', exc)

    log.info('Resolved CRS  : %s', resolved)
    return resolved, nominal_m


def build_grid_params(roi, grid_crs, nominal_m):
    '''Compute crs_transform + shape_2d for the actual ROI subset.

    Returns the dict that XEE's open_dataset() expects.
    The crs_transform describes THIS subset, not the global grid (FIX 18).
    '''
    minx, miny, maxx, maxy = roi.bounds

    transformer = pyproj.Transformer.from_crs(
        'EPSG:4326', grid_crs, always_xy=True)
    x1, y1 = transformer.transform(minx, miny)
    x2, y2 = transformer.transform(maxx, maxy)

    left   = min(x1, x2)
    right  = max(x1, x2)
    bottom = min(y1, y2)
    top    = max(y1, y2)

    try:
        unit = pyproj.CRS.from_user_input(grid_crs).axis_info[0].unit_name
    except Exception:
        unit = 'degree'

    if unit == 'metre':
        scale_x = float(nominal_m)
        scale_y = float(nominal_m)
    else:
        lat_centroid = roi.centroid.y
        cos_lat = np.cos(np.radians(lat_centroid))
        scale_y = nominal_m / 111_320.0
        scale_x = scale_y / cos_lat if cos_lat > 0.01 else scale_y

    width  = int(np.ceil((right - left) / scale_x))
    height = int(np.ceil((top - bottom) / scale_y))

    crs_transform = (scale_x, 0.0, left, 0.0, -scale_y, top)

    log.info('Grid : %d x %d px  |  scale_x=%.6g  scale_y=%.6g %s',
             width, height, scale_x, scale_y, unit)

    return {
        'crs': grid_crs,
        'crs_transform': crs_transform,
        'shape_2d': (width, height),
    }


# ─── Band Metadata Detection (FIX 20) ───────────────────────────────

def detect_band_types(collection, bands):
    '''Auto-detect native dtypes from GEE ee.Image.bandTypes().

    Returns {band_name: {'dtype': 'int16'|'uint16'|'float32'|...}}.
    '''
    detected = {}
    try:
        img = collection.select(0)
        bt = img.bandTypes().getInfo()
        for band_name in bands:
            if band_name not in bt:
                continue
            info = bt[band_name]
            precision = info.get('precision', 'float')
            mn = info.get('min', 0)
            mx = info.get('max', 0)

            if precision == 'int':
                if mn >= 0 and mx <= 255:
                    dtype = 'uint8'
                elif mn >= 0 and mx <= 65535:
                    dtype = 'uint16'
                elif mn >= -32768 and mx <= 32767:
                    dtype = 'int16'
                else:
                    dtype = 'int32'
            elif precision == 'double':
                dtype = 'float64'
            else:
                dtype = 'float32'

            detected[band_name] = {'dtype': dtype}
            log.info('  Band %-20s : %s (GEE: %s, range [%s, %s])',
                     band_name, dtype, precision, mn, mx)
    except Exception as e:
        log.warning('Cannot auto-detect band types: %s', e)

    return detected


def merge_band_info(bands, user_metadata, auto_detected):
    '''Merge user-provided BAND_METADATA with auto-detected types.

    User metadata takes precedence.  Auto-detected provides dtype fallback.
    Bands with no metadata at all get float32 + warning.
    '''
    result = {}
    warned = False

    for band in bands:
        info = {}

        # Layer 1: auto-detected dtype
        if band in auto_detected:
            info.update(auto_detected[band])

        # Layer 2: user-provided overrides everything
        if band in user_metadata:
            info.update(user_metadata[band])

        # Ensure dtype exists
        if 'dtype' not in info:
            info['dtype'] = 'float32'

        # Default _FillValue based on dtype if not user-specified
        if '_FillValue' not in info:
            dt = np.dtype(info['dtype'])
            if np.issubdtype(dt, np.unsignedinteger):
                info['_FillValue'] = int(np.iinfo(dt).max)       # e.g. 65535
            elif np.issubdtype(dt, np.signedinteger):
                info['_FillValue'] = int(np.iinfo(dt).min)       # e.g. -32768
            else:
                info['_FillValue'] = None   # NaN for floats (handled by encoding)

        # Warn about missing scale_factor (only once)
        if 'scale_factor' not in info and not warned:
            if band not in user_metadata:
                log.warning(
                    'BAND_METADATA missing for "%s" (and possibly others). '
                    'Raw values will be written without scale_factor/add_offset. '
                    'Populate BAND_METADATA from the GEE catalog for proper CF decoding.',
                    band)
                warned = True

        result[band] = info

    return result


# ─── Encoding & Metadata ────────────────────────────────────────────

def strip_xee_encoding(ds):
    '''Clear ALL XEE encoding (the bogus CRS-derived scale_factor).

    GEE already returns raw pixel values via computePixels.  XEE injects
    the CRS pixel size as scale_factor, which is NOT a CF data-packing
    parameter.  Clearing it prevents xarray from mis-interpreting it.
    '''
    for name in list(ds.data_vars) + list(ds.coords):
        if name in ds:
            ds[name].encoding.clear()
    return ds


def apply_band_metadata(ds, band_info):
    '''Write scale_factor, add_offset, valid_range, units, long_name
    as VARIABLE ATTRIBUTES — not encoding.

    FIX(16): scale_factor/add_offset are CF variable attributes.
    xarray's encoding dict interprets them as packing instructions
    (it would divide data by scale_factor before writing).  Since our
    data is ALREADY packed as raw integers from GEE, we must NOT put
    scale/offset in the encoding.  Writing them as attrs means:
      - xarray writes the raw values as-is
      - scale_factor appears in the file as a variable attribute
      - any CF reader (xarray, CDO, ncview) decodes automatically

    FIX(17): Per-band _FillValue is set individually.
    '''
    for var in list(ds.data_vars):
        if var not in band_info:
            continue
        info = band_info[var]

        # CF scale/offset as attributes
        if 'scale_factor' in info:
            ds[var].attrs['scale_factor'] = np.float64(info['scale_factor'])
        if 'add_offset' in info:
            ds[var].attrs['add_offset'] = np.float64(info['add_offset'])

        # Optional CF attributes
        if 'valid_range' in info:
            ds[var].attrs['valid_range'] = np.array(
                info['valid_range'], dtype=info.get('dtype', 'float32'))
        if 'long_name' in info:
            ds[var].attrs['long_name'] = info['long_name']
        if 'units' in info:
            ds[var].attrs['units'] = info['units']
        if 'flag_meanings' in info:
            ds[var].attrs['flag_meanings'] = info['flag_meanings']

    return ds


def prepare_for_write(ds, band_info):
    '''Cast variables to their target dtype, filling NaN → _FillValue.

    FIX(19): preserves native integer dtypes instead of blanket float32.
    For int16/uint16 bands, NaN (a float concept) is replaced with the
    band's _FillValue before the cast to integer.

    All operations are lazy (dask) — no eager computation.
    '''
    for var in list(ds.data_vars):
        if var not in band_info:
            continue

        info = band_info[var]
        target_dtype = np.dtype(info['dtype'])
        fill_val = info.get('_FillValue')

        if np.issubdtype(target_dtype, np.integer):
            # Integer dtypes cannot represent NaN.
            # Replace NaN with the fill value, then cast.
            if fill_val is None:
                fill_val = int(np.iinfo(target_dtype).min)
            ds[var] = ds[var].fillna(fill_val).astype(target_dtype)
        else:
            # Float dtypes: NaN stays as NaN (natural missing indicator)
            ds[var] = ds[var].astype(target_dtype)

    return ds


def build_encoding(ds, band_info, compress_level=4, chunk_xy=512):
    '''Build NetCDF encoding dict with per-band dtype and _FillValue.

    FIX(3):  chunksizes capped to actual dims.
    FIX(17): per-band _FillValue.
    FIX(19): native dtype from band_info.

    IMPORTANT: scale_factor and add_offset are NOT in the encoding.
    They are variable attributes set by apply_band_metadata().
    '''
    enc = {}

    for var in ds.data_vars:
        info = band_info.get(var, {})
        dtype = info.get('dtype', 'float32')
        fill = info.get('_FillValue')

        # For float types with no explicit fill, use NaN
        if fill is None and np.issubdtype(np.dtype(dtype), np.floating):
            fill = np.float32(np.nan) if dtype == 'float32' else np.float64(np.nan)

        # Cast fill to the target dtype for HDF5 compatibility
        if fill is not None:
            try:
                fill = np.dtype(dtype).type(fill)
            except (ValueError, OverflowError):
                pass

        # Chunk sizes capped to actual dimensions
        var_chunks = []
        for dim in ds[var].dims:
            dim_size = ds[var].sizes[dim]
            if dim == 'time':
                var_chunks.append(min(1, dim_size))
            elif dim in ('x', 'y'):
                var_chunks.append(min(chunk_xy, dim_size))
            else:
                var_chunks.append(min(1, dim_size))

        enc[var] = {
            'dtype': dtype,
            '_FillValue': fill,
            'zlib': compress_level > 0,
            'complevel': compress_level,
            'chunksizes': tuple(var_chunks),
        }

    # Coordinate encoding
    for c in ds.coords:
        if c == 'time':
            enc[c] = {
                'dtype': 'int64',
                '_FillValue': None,
                'units': 'days since 1970-01-01',
                'calendar': 'proleptic_gregorian',
            }
        elif c in ('x', 'y'):
            enc[c] = {'dtype': 'float64', '_FillValue': None}

    return enc


# ─── CF Grid Mapping (FIX 18) ───────────────────────────────────────

def add_grid_mapping(ds, grid_crs, crs_transform, grid_shape):
    '''Add a CF-compliant grid_mapping variable (spatial_ref).

    FIX(18): The file previously had no grid_mapping despite declaring
    CF-1.8, and the CRS metadata described the global MODIS grid, not
    the actual ROI subset.

    This function creates a scalar `spatial_ref` variable with:
      - grid_mapping_name + all CF projection parameters (via pyproj)
      - Full CRS WKT for precision
      - GeoTransform describing THIS subset (not the global grid)
      - Actual grid dimensions
    Every data variable gets a grid_mapping='spatial_ref' attribute.
    '''
    crs_obj = pyproj.CRS.from_user_input(grid_crs)

    # pyproj.CRS.to_cf() returns all required CF parameters:
    #   grid_mapping_name, semi_major_axis, inverse_flattening, etc.
    cf_attrs = crs_obj.to_cf()

    # Add full WKT for tools that prefer it (GDAL, QGIS, rioxarray)
    wkt = crs_obj.to_wkt()
    cf_attrs['crs_wkt']     = wkt
    cf_attrs['spatial_ref'] = wkt   # GDAL convention

    # GeoTransform for THIS subset (GDAL convention):
    #   (x_origin, x_pixel_size, x_rotation, y_origin, y_rotation, y_pixel_size)
    # Our crs_transform: (x_scale, 0, x_origin, 0, -y_scale, y_origin)
    geo_transform = (
        f'{crs_transform[2]} {crs_transform[0]} {crs_transform[1]} '
        f'{crs_transform[5]} {crs_transform[3]} {crs_transform[4]}'
    )
    cf_attrs['GeoTransform'] = geo_transform

    # Grid dimensions of the actual subset
    width, height = grid_shape
    cf_attrs['grid_width']  = int(width)
    cf_attrs['grid_height'] = int(height)

    # Create the scalar grid_mapping variable
    ds['spatial_ref'] = xr.DataArray(
        data=np.int32(0),
        attrs=cf_attrs,
    )

    # Tag every data variable with grid_mapping
    for var in list(ds.data_vars):
        if var != 'spatial_ref':
            ds[var].attrs['grid_mapping'] = 'spatial_ref'

    return ds


def make_cf_compliant(ds, dataset_id, roi_label, grid_crs,
                      crs_transform, grid_shape, year):
    '''Stamp CF-1.8 global + coordinate attributes + grid_mapping.

    FIX(18): now includes grid_mapping variable and subset-specific
    GeoTransform instead of global grid metadata.
    '''
    width, height = grid_shape

    ds.attrs.update({
        'Conventions': 'CF-1.8',
        'title':  f'{dataset_id} -- {roi_label} ({year})',
        'source': f'Google Earth Engine: {dataset_id}',
        'history': f'Created {datetime.now(timezone.utc).isoformat()}',
        'crs': grid_crs,
        'geospatial_bounds_crs': grid_crs,
        'grid_width': int(width),
        'grid_height': int(height),
    })

    if 'time' in ds.coords:
        ds['time'].attrs.update(axis='T', standard_name='time')

    try:
        is_proj = (pyproj.CRS.from_user_input(grid_crs)
                   .axis_info[0].unit_name == 'metre')
    except Exception:
        is_proj = False

    if 'x' in ds.coords:
        ds['x'].attrs.update(
            axis='X',
            standard_name='projection_x_coordinate' if is_proj else 'longitude',
            units='m' if is_proj else 'degrees_east',
        )
    if 'y' in ds.coords:
        ds['y'].attrs.update(
            axis='Y',
            standard_name='projection_y_coordinate' if is_proj else 'latitude',
            units='m' if is_proj else 'degrees_north',
        )

    # Add grid_mapping variable
    ds = add_grid_mapping(ds, grid_crs, crs_transform, grid_shape)

    return ds


def sanitize_attrs(ds):
    '''Flatten complex GEE metadata; drop None values.'''
    for var in list(ds.data_vars) + list(ds.coords):
        if var not in ds:
            continue
        to_drop = []
        for key, val in ds[var].attrs.items():
            if val is None:
                to_drop.append(key)
            elif not isinstance(val, (str, int, float, np.number, np.ndarray)):
                ds[var].attrs[key] = str(val)
        for key in to_drop:
            del ds[var].attrs[key]

    to_drop = []
    for key, val in ds.attrs.items():
        if val is None:
            to_drop.append(key)
        elif not isinstance(val, (str, int, float, np.number, np.ndarray)):
            ds.attrs[key] = str(val)
    for key in to_drop:
        del ds.attrs[key]

    return ds


# ─── Pre-Write Validation ──────────────────────────────────────────
import random

# ─── Pre-Write Validation ──────────────────────────────────────────

def pre_write_check(ds, year, n_samples=25, block_size=100):
    '''Monte Carlo spatial sampling to check for valid data.

    Randomly drops N blocks of 50x50 pixels across the grid.
    Prevents downloading 1.3 billion pixels just to verify integrity.
    '''
    data_vars = [v for v in ds.data_vars if v != 'spatial_ref']
    if not data_vars:
        return

    var_name = data_vars[0]
    da = ds[var_name]

    nx = da.sizes.get('x', 0)
    ny = da.sizes.get('y', 0)

    if nx == 0 or ny == 0:
        return

    # Adjust block size if grid is smaller than 50px
    bs = min(block_size, nx, ny)

    valid_found = False
    for i in range(n_samples):
        # Pick a random top-left corner
        rand_x = random.randint(0, nx - bs)
        rand_y = random.randint(0, ny - bs)

        indexers = {
            'x': slice(rand_x, rand_x + bs),
            'y': slice(rand_y, rand_y + bs)
        }
        if 'time' in da.dims:
            indexers['time'] = 0

        # Compute ONLY this tiny 50x50 block
        sample = da.isel(**indexers).compute()
        vals = sample.values

        if np.issubdtype(vals.dtype, np.floating):
            if np.isfinite(vals).sum() > 0:
                valid_found = True
                break
        else:
            fill = da.attrs.get('_FillValue', da.encoding.get('_FillValue', None))
            if fill is not None:
                if (vals != fill).sum() > 0:
                    valid_found = True
                    break
            else:
                valid_found = True
                break

    if not valid_found:
        raise DataQualityError(
            f'[{year}] Monte Carlo pre-check failed: No valid pixels found '
            f'in {n_samples} random samples. Check ROI/CRS.')

    log.info('[%d] Pre-check OK (Monte Carlo: %d random blocks sampled)', year, i + 1)


# ─── Post-Write Integrity Verification ─────────────────────────────

def post_write_verify(filepath, expected_vars, band_info, year):
    '''Reopen the NetCDF and verify structure + data integrity.

    FIX(11): checks ALL bands.
    FIX(16): now expects legitimate scale_factor on bands that declare it
             in BAND_METADATA (no longer flags them as "bogus").
    '''
    try:
        vds = xr.open_dataset(filepath, chunks='auto', mask_and_scale=False)
    except Exception as e:
        return False, f'Cannot reopen: {e}'

    try:
        missing = [v for v in expected_vars if v not in vds.data_vars]
        if missing:
            return False, f'Missing vars: {missing}'

        if 'time' not in vds.dims:
            return False, 'No time dimension'

        # grid_mapping present?
        if 'spatial_ref' not in vds:
            return False, 'Missing spatial_ref grid_mapping variable'

        # Coordinates finite?
        for c in ('x', 'y'):
            if c in vds.coords:
                vals = vds[c].values
                if np.issubdtype(vals.dtype, np.floating):
                    if not np.all(np.isfinite(vals)):
                        return False, f'Non-finite coord: {c}'

        # Check scale_factor: should match BAND_METADATA (not CRS pixel size)
        for v in expected_vars:
            info = band_info.get(v, {})
            expected_sf = info.get('scale_factor')
            actual_sf = vds[v].attrs.get('scale_factor',
                                         vds[v].encoding.get('scale_factor'))

            if expected_sf is not None:
                if actual_sf is None:
                    return False, f'{v}: expected scale_factor={expected_sf}, got none'
                if abs(float(actual_sf) - float(expected_sf)) > 1e-10:
                    return False, (
                        f'{v}: scale_factor={actual_sf} != expected {expected_sf} '
                        '(possible XEE pollution)')
            else:
                # Band with no expected scale — should have none
                if actual_sf is not None and float(actual_sf) != 1.0:
                    return False, (
                        f'{v}: unexpected scale_factor={actual_sf}')

        # Data has valid values? (sample ALL variables)
        bad_vars = []
        for v in expected_vars:
            da = vds[v]
            info = band_info.get(v, {})
            fill_val = info.get('_FillValue')

            idx = {}
            if 'time' in da.dims:
                idx['time'] = 0
            for d in da.dims:
                if d != 'time':
                    s = da.sizes[d]
                    m = s // 2
                    hw = max(1, min(50, s // 4))
                    idx[d] = slice(max(0, m - hw), min(s, m + hw))

            samp = da.isel(**idx).compute().values

            if np.issubdtype(samp.dtype, np.floating):
                valid = np.any(np.isfinite(samp))
            else:
                if fill_val is not None:
                    valid = np.any(samp != fill_val)
                else:
                    valid = samp.size > 0

            if not valid:
                bad_vars.append(v)

        if bad_vars:
            return False, f'No valid data on re-read: {bad_vars}'

        return True, f'All {len(expected_vars)} vars verified with correct metadata'

    finally:
        vds.close()


# ─── Drive Copy ─────────────────────────────────────────────────────

def get_md5(filepath, chunk_bytes=10 * 1024 * 1024):
    '''MD5 with standard buffered I/O (no O_DIRECT — FUSE incompatible).'''
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    '''Copy to Drive, flush, verify MD5.'''
    src_md5 = get_md5(src)

    for attempt in range(1, max_retries + 1):
        try:
            log.info('Drive copy attempt %d/%d ...', attempt, max_retries)

            with open(src, 'rb') as f_src, open(dst, 'wb') as f_dst:
                shutil.copyfileobj(f_src, f_dst, length=10 * 1024 * 1024)
                f_dst.flush()
                os.fsync(f_dst.fileno())

            os.sync()
            time.sleep(3)

            src_size = os.path.getsize(src)
            dst_size = os.path.getsize(dst)
            if src_size != dst_size:
                raise IOError(
                    f'Size mismatch: src={src_size}, dst={dst_size}')

            dst_md5 = get_md5(dst)
            if src_md5 != dst_md5:
                raise IOError('MD5 mismatch')

            log.info('Drive copy verified (MD5 match, %d bytes)', src_size)
            return True

        except Exception as e:
            log.error('Drive copy attempt %d failed: %s', attempt, e)
            if os.path.exists(dst):
                try:
                    os.remove(dst)
                except OSError:
                    pass
            if attempt < max_retries:
                wait = 15 * attempt
                log.info('Retrying in %ds ...', wait)
                time.sleep(wait)
            else:
                raise DataQualityError(
                    f'Drive copy failed after {max_retries} attempts: {e}')


# ╔════════════════════════════════════════════════════════════════════╗
# ║  MAIN PIPELINE                                                     ║
# ╚════════════════════════════════════════════════════════════════════╝

def run_pipeline():
    log.info('=' * 62)
    log.info('  Dataset : %s', DATASET_ID)
    log.info('=' * 62)

    # ── 1. Collection metadata ──────────────────────────────────
    coll = ee.Image(DATASET_ID)
    bands = coll.bandNames().getInfo()
    if not bands:
        raise ValueError(f'{DATASET_ID} returned no bands')

    years = [2019]

    # years = list(range(y0, y1 + 1))
    # log.info('Bands (%d) : %s', len(bands), bands)
    # log.info('Years      : %d - %d  (%d total)', y0, y1, len(years))

    # ── 2. CRS & grid ──────────────────────────────────────────
    grid_crs, nominal_m = resolve_native_crs(coll.select(bands))

    tolerance_deg = (nominal_m * 1.5) / 111_320.0
    roi = load_roi(SHAPEFILE_DIR, simplify_deg=tolerance_deg)

    short = DATASET_ID.split('/')[-1]
    out_dir = os.path.join(DRIVE_ROOT, short)
    os.makedirs(out_dir, exist_ok=True)
    log.info('Output : %s', out_dir)

    grid_params = build_grid_params(roi, grid_crs, nominal_m)
    crs_transform = grid_params['crs_transform']
    grid_shape    = grid_params['shape_2d']

    # ── 3. Band metadata (auto-detect + user override) ──────────
    log.info('Detecting band types from GEE ...')
    auto_types = detect_band_types(coll.select(bands), bands)
    band_info  = merge_band_info(bands, BAND_METADATA, auto_types)

    log.info('Band info summary:')
    for b in bands:
        info = band_info[b]
        sf = info.get('scale_factor', '—')
        fv = info.get('_FillValue', '—')
        log.info('  %-20s  dtype=%-7s  scale=%-8s  fill=%s',
                 b, info['dtype'], sf, fv)

    # ── 4. Year loop ────────────────────────────────────────────
    n_ok = n_fail = n_skip = 0

    for year in years:
        fname = f'{short}_{year}_{ROI_LABEL}.nc'
        final = os.path.join(out_dir, fname)

        # ── Resume check ────────────────────────────────────────
        if os.path.exists(final) and not FORCE_OVERWRITE:
            sz = os.path.getsize(final)
            if sz < 1024:
                log.warning('[%d] Tiny file (%d B) — re-processing', year, sz)
                os.remove(final)
            else:
                is_valid = False
                try:
                    with xr.open_dataset(final, mask_and_scale=False) as chk:
                        if ('time' in chk.dims and chk.sizes['time'] > 0
                                and 'spatial_ref' in chk):
                            is_valid = True
                except Exception:
                    pass

                if is_valid:
                    log.info('[%d] On Drive (%.1f MiB), valid — skip',
                             year, sz / 1024**2)
                    n_skip += 1
                    continue
                else:
                    log.warning('[%d] Existing file missing metadata — re-processing', year)
                    try:
                        os.remove(final)
                    except OSError:
                        pass

        tmp = os.path.join('/content', fname)
        done = False

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                log.info('[%d] attempt %d/%d', year, attempt, MAX_RETRIES)

                # A static ee.Image has no time axis.  Wrap it in a 1-element
                # ImageCollection and stamp a synthetic system:time_start so that
                # xee creates a length-1 'time' dimension (the rest of the pipeline
                # assumes 'time' exists).
                yr_col = ee.ImageCollection([
                    coll.select(bands).set(
                        'system:time_start',
                        ee.Date(f'{year}-01-01').millis()
                    )
                ])
                log.info('[%d] 1 image (wrapped ee.Image)', year)

                # ── Open lazily ──────────────────────────────────
                grid_w, grid_h = grid_shape
                safe_chunk = max(16, min(CHUNK_XY, grid_w, grid_h))

                ds = xr.open_dataset(
                    yr_col,
                    engine='ee',
                    chunks={'x': safe_chunk, 'y': safe_chunk},
                    **grid_params,
                )

                # Deduplicate timestamps
                _, unique_idx = np.unique(ds['time'], return_index=True)
                if len(unique_idx) != ds.sizes['time']:
                    log.warning('[%d] Deduplicating timestamps', year)
                    ds = ds.isel(time=sorted(unique_idx))

                # ── Fix encoding ─────────────────────────────────
                ds = strip_xee_encoding(ds)

                # ── Apply band metadata (scale, fill, units) ─────
                ds = apply_band_metadata(ds, band_info)

                # ── Cast to native dtypes (NaN → fill for ints) ──
                ds = prepare_for_write(ds, band_info)

                # ── CF compliance + grid_mapping ─────────────────
                ds = sanitize_attrs(ds)
                ds = make_cf_compliant(
                    ds, DATASET_ID, ROI_LABEL, grid_crs,
                    crs_transform, grid_shape, year)

                # ── Pre-write validation ─────────────────────────
                pre_write_check(ds, year)

                # ── Write to scratch ─────────────────────────────
                enc = build_encoding(ds, band_info, COMPRESS_LEVEL, safe_chunk)

                log.info('[%d] Writing -> %s', year, tmp)
                t0 = time.time()

                try:
                    with dask.config.set(scheduler='synchronous'):
                        ds.to_netcdf(
                            tmp,
                            engine='netcdf4',
                            encoding=enc,
                            unlimited_dims=['time'],
                        )
                except Exception:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                    raise

                dt = time.time() - t0
                mb = os.path.getsize(tmp) / 1024**2
                log.info('[%d] Written in %.0fs  (%.1f MiB)', year, dt, mb)

                if os.path.getsize(tmp) < 1024:
                    raise DataQualityError(
                        f'[{year}] File too small ({os.path.getsize(tmp)} B)')

                # ── Post-write integrity ─────────────────────────
                ok_flag, msg = post_write_verify(tmp, bands, band_info, year)
                if not ok_flag:
                    raise DataQualityError(
                        f'[{year}] INTEGRITY FAIL: {msg}')
                log.info('[%d] Integrity OK: %s', year, msg)

                # ── Export to Drive ───────────────────────────────
                log.info('[%d] Exporting to Drive ...', year)
                t_mv = time.time()
                robust_drive_copy(tmp, final, max_retries=3)
                os.remove(tmp)
                log.info('[%d] Exported in %.0fs -> %s',
                         year, time.time() - t_mv, final)

                n_ok += 1
                done = True
                break

            except DataQualityError as e:
                log.error('[%d] DATA ERROR (not retrying): %s', year, e)
                n_fail += 1
                break

            except Exception as e:
                log.error('[%d] attempt %d error: %s', year, attempt, e)
                if attempt < MAX_RETRIES:
                    wait = 30 * 2 ** (attempt - 1)
                    log.info('[%d] retry in %ds ...', year, wait)
                    time.sleep(wait)
                else:
                    log.error('[%d] retries exhausted', year)
                    n_fail += 1

            finally:

                if not done and os.path.exists(tmp):
                    try:
                        os.remove(tmp)
                    except OSError:
                        pass
                gc.collect()
                gc.collect()

    # ── Summary ─────────────────────────────────────────────────
    log.info('=' * 62)
    log.info('  DONE  %d ok | %d fail | %d skip  (%d years)',
             n_ok, n_fail, n_skip, len(years))
    log.info('=' * 62)

    if os.path.isdir(out_dir):
        nc_files = sorted(f for f in os.listdir(out_dir) if f.endswith('.nc'))
        if nc_files:
            log.info('Files on Drive (%s):', out_dir)
            for f in nc_files:
                sz = os.path.getsize(os.path.join(out_dir, f)) / 1024**2
                log.info('  %s  (%.1f MiB)', f, sz)


if __name__ == '__main__':
    run_pipeline()

# Water Table depth Ma et al
### "projects/sat-io/open-datasets/HRES-WTD"

In [ ]:
# ==========================================
# 1. Install Required Packages & Setup Logging
# ==========================================
!pip install -q xee xarray netcdf4 geopandas geemap pyproj dask
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
gee_to_netcdf_v3.py — Earth Engine → CF-Compliant NetCDF Exporter
===================================================================

Changes from v2
---------------
FIX(16) scale_factor/add_offset written per-band as CF variable attributes
        so raw packed values can be decoded by any CF-compliant reader.
FIX(17) Per-band _FillValue: each variable gets its own native fill value.
        QA bitmask bands get uint16 dtype; angle bands get scale=0.01; etc.
FIX(18) grid_mapping variable (`spatial_ref`) added to every file.
        Contains full CRS WKT, GeoTransform, and CF parameters.
        Every data variable references it via grid_mapping='spatial_ref'.
FIX(19) Native integer dtypes preserved (int16 for NDVI, uint16 for QA).
        v2 cast everything to float32, losing packing + fill semantics.
FIX(20) Auto-detect band dtypes from GEE when BAND_METADATA is empty.
        Falls back to the detected dtype, logs warning about missing scale.

All v2 crash/logic/integrity fixes (FIX 1-15) are carried forward.
"""

#
# ║  CONFIGURATION                                                     ║
# ╚════════════════════════════════════════════════════════════════════╝


DATASET_ID = "projects/sat-io/open-datasets/HRES-WTD"
PROJECT_ID = "msugw-503806"  # Keep your project ID

SHAPEFILE_DIR = "/content/ogallala_shp"
ROI_LABEL     = "Ogallala"
DRIVE_ROOT    = "/content/drive/MyDrive/MSUGWB"

# HRES-WTD by Ma et al. is typically a static/global snapshot or a specific year.
# If it's a single image, setting both to the same year ensures the pipeline
# processes it exactly once.
YEAR_START = 2015  # Adjust based on the dataset's actual publication/year
YEAR_END   = 2015

DASK_WORKERS    = 4
CHUNK_XY        = 2048
COMPRESS_LEVEL  = 0
MAX_RETRIES     = 3
FORCE_OVERWRITE = False

# ── Per-Band Metadata (HRES-WTD) ────────────────────────────────────
#
# Source: projects/sat-io/open-datasets/HRES-WTD
# Variables: Water Table Depth (WTD) in meters.
#
# Note: We include both "b1" and "wtd" below. The script will auto-detect
# the actual band name from GEE and apply whichever matches. If GEE returns
# "b1", it uses "b1"; if it returns "wtd", it uses "wtd".
#
BAND_METADATA = {
    "b1": {
        "dtype": "float32",
        "scale_factor": 1.0,
        "add_offset": 0.0,
        "_FillValue": None,
        "long_name": "Water Table Depth",
        "units": "meters"
    }
}


# ╔════════════════════════════════════════════════════════════════════╗
# ║  SETUP                                                             ║
# ╚════════════════════════════════════════════════════════════════════╝

# --- Colab prerequisites ---
# Uncomment the next two lines when running on Colab:
# !pip install -q "xee>=0.0.14" xarray netcdf4 geopandas pyproj
# from google.colab import drive; drive.mount('/content/drive')

import os, sys, glob, time, shutil, logging, gc, hashlib
from datetime import datetime, timezone

import numpy as np
import ee
import xarray as xr
import xee
from xee import helpers
import geopandas as gpd
import pyproj
import dask

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('gee_export')
logging.getLogger('urllib3.connectionpool').setLevel(logging.ERROR)

dask.config.set(scheduler='threads', num_workers=DASK_WORKERS)

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
log.info('Earth Engine ready  (project=%s)', PROJECT_ID)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  EXCEPTIONS                                                        ║
# ╚════════════════════════════════════════════════════════════════════╝

class DataQualityError(Exception):
    """Non-retryable data quality error."""
    pass


# ╔════════════════════════════════════════════════════════════════════╗
# ║  PIPELINE FUNCTIONS                                                ║
# ╚════════════════════════════════════════════════════════════════════╝

# ─── ROI ─────────────────────────────────────────────────────────────

def load_roi(shapefile_dir, simplify_deg=0.001):
    '''Load shapefile -> dissolved EPSG:4326 geometry.'''
    paths = glob.glob(os.path.join(shapefile_dir, '**', '*.shp'), recursive=True)
    if not paths:
        raise FileNotFoundError(f'No .shp found in {shapefile_dir}')
    log.info('Shapefile: %s', paths[0])

    gdf = gpd.read_file(paths[0])
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    roi = (gdf.geometry.union_all()
           if hasattr(gdf.geometry, 'union_all')
           else gdf.geometry.unary_union)

    try:
        import shapely as _shp
        nv = int(_shp.get_num_coordinates(roi))
    except Exception:
        nv = len(roi.wkt) // 20

    if nv > 50_000:
        roi = roi.simplify(simplify_deg, preserve_topology=True)
        try:
            nv_after = int(_shp.get_num_coordinates(roi))
        except Exception:
            nv_after = len(roi.wkt) // 20
        log.warning('Simplified ROI: %s -> %s vertices (tol=%.4f deg)',
                    f'{nv:,}', f'{nv_after:,}', simplify_deg)
    else:
        log.info('ROI vertices: %s', f'{nv:,}')
    return roi


# ─── CRS & Grid ─────────────────────────────────────────────────────

def resolve_native_crs(collection):
    '''Detect native CRS + nominal scale.  Returns (epsg_str, metres).'''
    proj = collection.first().select(0).projection()
    info = proj.getInfo()
    native_crs = info.get('crs', 'EPSG:4326')
    nominal_m  = proj.nominalScale().getInfo()

    log.info('Native CRS    : %s', native_crs[:70])
    log.info('Nominal scale : %.1f m', nominal_m)

    resolved = 'EPSG:4326'

    if native_crs.startswith('SR-ORG:'):
        log.warning('SR-ORG CRS (%s) -> EPSG:4326 fallback', native_crs)
    elif native_crs.startswith('EPSG:'):
        resolved = native_crs
    else:
        try:
            crs_obj = pyproj.CRS.from_user_input(native_crs)
            epsg = crs_obj.to_epsg()
            if epsg:
                resolved = f'EPSG:{epsg}'
            elif crs_obj.axis_info and crs_obj.axis_info[0].unit_name == 'metre':
                resolved = 'EPSG:5070'
                log.warning('Metre CRS, no EPSG -> EPSG:5070')
        except Exception as exc:
            log.warning('CRS parse error: %s -> EPSG:4326', exc)

    log.info('Resolved CRS  : %s', resolved)
    return resolved, nominal_m


def build_grid_params(roi, grid_crs, nominal_m):
    '''Compute crs_transform + shape_2d for the actual ROI subset.

    Returns the dict that XEE's open_dataset() expects.
    The crs_transform describes THIS subset, not the global grid (FIX 18).
    '''
    minx, miny, maxx, maxy = roi.bounds

    transformer = pyproj.Transformer.from_crs(
        'EPSG:4326', grid_crs, always_xy=True)
    x1, y1 = transformer.transform(minx, miny)
    x2, y2 = transformer.transform(maxx, maxy)

    left   = min(x1, x2)
    right  = max(x1, x2)
    bottom = min(y1, y2)
    top    = max(y1, y2)

    try:
        unit = pyproj.CRS.from_user_input(grid_crs).axis_info[0].unit_name
    except Exception:
        unit = 'degree'

    if unit == 'metre':
        scale_x = float(nominal_m)
        scale_y = float(nominal_m)
    else:
        lat_centroid = roi.centroid.y
        cos_lat = np.cos(np.radians(lat_centroid))
        scale_y = nominal_m / 111_320.0
        scale_x = scale_y / cos_lat if cos_lat > 0.01 else scale_y

    width  = int(np.ceil((right - left) / scale_x))
    height = int(np.ceil((top - bottom) / scale_y))

    crs_transform = (scale_x, 0.0, left, 0.0, -scale_y, top)

    log.info('Grid : %d x %d px  |  scale_x=%.6g  scale_y=%.6g %s',
             width, height, scale_x, scale_y, unit)

    return {
        'crs': grid_crs,
        'crs_transform': crs_transform,
        'shape_2d': (width, height),
    }


# ─── Band Metadata Detection (FIX 20) ───────────────────────────────

def detect_band_types(collection, bands):
    '''Auto-detect native dtypes from GEE ee.Image.bandTypes().

    Returns {band_name: {'dtype': 'int16'|'uint16'|'float32'|...}}.
    '''
    detected = {}
    try:
        img = collection.first()
        bt = img.bandTypes().getInfo()
        for band_name in bands:
            if band_name not in bt:
                continue
            info = bt[band_name]
            precision = info.get('precision', 'float')
            mn = info.get('min', 0)
            mx = info.get('max', 0)

            if precision == 'int':
                if mn >= 0 and mx <= 255:
                    dtype = 'uint8'
                elif mn >= 0 and mx <= 65535:
                    dtype = 'uint16'
                elif mn >= -32768 and mx <= 32767:
                    dtype = 'int16'
                else:
                    dtype = 'int32'
            elif precision == 'double':
                dtype = 'float64'
            else:
                dtype = 'float32'

            detected[band_name] = {'dtype': dtype}
            log.info('  Band %-20s : %s (GEE: %s, range [%s, %s])',
                     band_name, dtype, precision, mn, mx)
    except Exception as e:
        log.warning('Cannot auto-detect band types: %s', e)

    return detected


def merge_band_info(bands, user_metadata, auto_detected):
    '''Merge user-provided BAND_METADATA with auto-detected types.

    User metadata takes precedence.  Auto-detected provides dtype fallback.
    Bands with no metadata at all get float32 + warning.
    '''
    result = {}
    warned = False

    for band in bands:
        info = {}

        # Layer 1: auto-detected dtype
        if band in auto_detected:
            info.update(auto_detected[band])

        # Layer 2: user-provided overrides everything
        if band in user_metadata:
            info.update(user_metadata[band])

        # Ensure dtype exists
        if 'dtype' not in info:
            info['dtype'] = 'float32'

        # Default _FillValue based on dtype if not user-specified
        if '_FillValue' not in info:
            dt = np.dtype(info['dtype'])
            if np.issubdtype(dt, np.unsignedinteger):
                info['_FillValue'] = int(np.iinfo(dt).max)       # e.g. 65535
            elif np.issubdtype(dt, np.signedinteger):
                info['_FillValue'] = int(np.iinfo(dt).min)       # e.g. -32768
            else:
                info['_FillValue'] = None   # NaN for floats (handled by encoding)

        # Warn about missing scale_factor (only once)
        if 'scale_factor' not in info and not warned:
            if band not in user_metadata:
                log.warning(
                    'BAND_METADATA missing for "%s" (and possibly others). '
                    'Raw values will be written without scale_factor/add_offset. '
                    'Populate BAND_METADATA from the GEE catalog for proper CF decoding.',
                    band)
                warned = True

        result[band] = info

    return result


# ─── Encoding & Metadata ────────────────────────────────────────────

def strip_xee_encoding(ds):
    '''Clear ALL XEE encoding (the bogus CRS-derived scale_factor).

    GEE already returns raw pixel values via computePixels.  XEE injects
    the CRS pixel size as scale_factor, which is NOT a CF data-packing
    parameter.  Clearing it prevents xarray from mis-interpreting it.
    '''
    for name in list(ds.data_vars) + list(ds.coords):
        if name in ds:
            ds[name].encoding.clear()
    return ds


def apply_band_metadata(ds, band_info):
    '''Write scale_factor, add_offset, valid_range, units, long_name
    as VARIABLE ATTRIBUTES — not encoding.

    FIX(16): scale_factor/add_offset are CF variable attributes.
    xarray's encoding dict interprets them as packing instructions
    (it would divide data by scale_factor before writing).  Since our
    data is ALREADY packed as raw integers from GEE, we must NOT put
    scale/offset in the encoding.  Writing them as attrs means:
      - xarray writes the raw values as-is
      - scale_factor appears in the file as a variable attribute
      - any CF reader (xarray, CDO, ncview) decodes automatically

    FIX(17): Per-band _FillValue is set individually.
    '''
    for var in list(ds.data_vars):
        if var not in band_info:
            continue
        info = band_info[var]

        # CF scale/offset as attributes
        if 'scale_factor' in info:
            ds[var].attrs['scale_factor'] = np.float64(info['scale_factor'])
        if 'add_offset' in info:
            ds[var].attrs['add_offset'] = np.float64(info['add_offset'])

        # Optional CF attributes
        if 'valid_range' in info:
            ds[var].attrs['valid_range'] = np.array(
                info['valid_range'], dtype=info.get('dtype', 'float32'))
        if 'long_name' in info:
            ds[var].attrs['long_name'] = info['long_name']
        if 'units' in info:
            ds[var].attrs['units'] = info['units']
        if 'flag_meanings' in info:
            ds[var].attrs['flag_meanings'] = info['flag_meanings']

    return ds


def prepare_for_write(ds, band_info):
    '''Cast variables to their target dtype, filling NaN → _FillValue.

    FIX(19): preserves native integer dtypes instead of blanket float32.
    For int16/uint16 bands, NaN (a float concept) is replaced with the
    band's _FillValue before the cast to integer.

    All operations are lazy (dask) — no eager computation.
    '''
    for var in list(ds.data_vars):
        if var not in band_info:
            continue

        info = band_info[var]
        target_dtype = np.dtype(info['dtype'])
        fill_val = info.get('_FillValue')

        if np.issubdtype(target_dtype, np.integer):
            # Integer dtypes cannot represent NaN.
            # Replace NaN with the fill value, then cast.
            if fill_val is None:
                fill_val = int(np.iinfo(target_dtype).min)
            ds[var] = ds[var].fillna(fill_val).astype(target_dtype)
        else:
            # Float dtypes: NaN stays as NaN (natural missing indicator)
            ds[var] = ds[var].astype(target_dtype)

    return ds


def build_encoding(ds, band_info, compress_level=4, chunk_xy=512):
    '''Build NetCDF encoding dict with per-band dtype and _FillValue.

    FIX(3):  chunksizes capped to actual dims.
    FIX(17): per-band _FillValue.
    FIX(19): native dtype from band_info.

    IMPORTANT: scale_factor and add_offset are NOT in the encoding.
    They are variable attributes set by apply_band_metadata().
    '''
    enc = {}

    for var in ds.data_vars:
        info = band_info.get(var, {})
        dtype = info.get('dtype', 'float32')
        fill = info.get('_FillValue')

        # For float types with no explicit fill, use NaN
        if fill is None and np.issubdtype(np.dtype(dtype), np.floating):
            fill = np.float32(np.nan) if dtype == 'float32' else np.float64(np.nan)

        # Cast fill to the target dtype for HDF5 compatibility
        if fill is not None:
            try:
                fill = np.dtype(dtype).type(fill)
            except (ValueError, OverflowError):
                pass

        # Chunk sizes capped to actual dimensions
        var_chunks = []
        for dim in ds[var].dims:
            dim_size = ds[var].sizes[dim]
            if dim == 'time':
                var_chunks.append(min(1, dim_size))
            elif dim in ('x', 'y'):
                var_chunks.append(min(chunk_xy, dim_size))
            else:
                var_chunks.append(min(1, dim_size))

        enc[var] = {
            'dtype': dtype,
            '_FillValue': fill,
            'zlib': compress_level > 0,
            'complevel': compress_level,
            'chunksizes': tuple(var_chunks),
        }

    # Coordinate encoding
    for c in ds.coords:
        if c == 'time':
            enc[c] = {
                'dtype': 'int64',
                '_FillValue': None,
                'units': 'days since 1970-01-01',
                'calendar': 'proleptic_gregorian',
            }
        elif c in ('x', 'y'):
            enc[c] = {'dtype': 'float64', '_FillValue': None}

    return enc


# ─── CF Grid Mapping (FIX 18) ───────────────────────────────────────

def add_grid_mapping(ds, grid_crs, crs_transform, grid_shape):
    '''Add a CF-compliant grid_mapping variable (spatial_ref).

    FIX(18): The file previously had no grid_mapping despite declaring
    CF-1.8, and the CRS metadata described the global MODIS grid, not
    the actual ROI subset.

    This function creates a scalar `spatial_ref` variable with:
      - grid_mapping_name + all CF projection parameters (via pyproj)
      - Full CRS WKT for precision
      - GeoTransform describing THIS subset (not the global grid)
      - Actual grid dimensions
    Every data variable gets a grid_mapping='spatial_ref' attribute.
    '''
    crs_obj = pyproj.CRS.from_user_input(grid_crs)

    # pyproj.CRS.to_cf() returns all required CF parameters:
    #   grid_mapping_name, semi_major_axis, inverse_flattening, etc.
    cf_attrs = crs_obj.to_cf()

    # Add full WKT for tools that prefer it (GDAL, QGIS, rioxarray)
    wkt = crs_obj.to_wkt()
    cf_attrs['crs_wkt']     = wkt
    cf_attrs['spatial_ref'] = wkt   # GDAL convention

    # GeoTransform for THIS subset (GDAL convention):
    #   (x_origin, x_pixel_size, x_rotation, y_origin, y_rotation, y_pixel_size)
    # Our crs_transform: (x_scale, 0, x_origin, 0, -y_scale, y_origin)
    geo_transform = (
        f'{crs_transform[2]} {crs_transform[0]} {crs_transform[1]} '
        f'{crs_transform[5]} {crs_transform[3]} {crs_transform[4]}'
    )
    cf_attrs['GeoTransform'] = geo_transform

    # Grid dimensions of the actual subset
    width, height = grid_shape
    cf_attrs['grid_width']  = int(width)
    cf_attrs['grid_height'] = int(height)

    # Create the scalar grid_mapping variable
    ds['spatial_ref'] = xr.DataArray(
        data=np.int32(0),
        attrs=cf_attrs,
    )

    # Tag every data variable with grid_mapping
    for var in list(ds.data_vars):
        if var != 'spatial_ref':
            ds[var].attrs['grid_mapping'] = 'spatial_ref'

    return ds


def make_cf_compliant(ds, dataset_id, roi_label, grid_crs,
                      crs_transform, grid_shape, year):
    '''Stamp CF-1.8 global + coordinate attributes + grid_mapping.

    FIX(18): now includes grid_mapping variable and subset-specific
    GeoTransform instead of global grid metadata.
    '''
    width, height = grid_shape

    ds.attrs.update({
        'Conventions': 'CF-1.8',
        'title':  f'{dataset_id} -- {roi_label} ({year})',
        'source': f'Google Earth Engine: {dataset_id}',
        'history': f'Created {datetime.now(timezone.utc).isoformat()}',
        'crs': grid_crs,
        'geospatial_bounds_crs': grid_crs,
        'grid_width': int(width),
        'grid_height': int(height),
    })

    if 'time' in ds.coords:
        ds['time'].attrs.update(axis='T', standard_name='time')

    try:
        is_proj = (pyproj.CRS.from_user_input(grid_crs)
                   .axis_info[0].unit_name == 'metre')
    except Exception:
        is_proj = False

    if 'x' in ds.coords:
        ds['x'].attrs.update(
            axis='X',
            standard_name='projection_x_coordinate' if is_proj else 'longitude',
            units='m' if is_proj else 'degrees_east',
        )
    if 'y' in ds.coords:
        ds['y'].attrs.update(
            axis='Y',
            standard_name='projection_y_coordinate' if is_proj else 'latitude',
            units='m' if is_proj else 'degrees_north',
        )

    # Add grid_mapping variable
    ds = add_grid_mapping(ds, grid_crs, crs_transform, grid_shape)

    return ds


def sanitize_attrs(ds):
    '''Flatten complex GEE metadata; drop None values.'''
    for var in list(ds.data_vars) + list(ds.coords):
        if var not in ds:
            continue
        to_drop = []
        for key, val in ds[var].attrs.items():
            if val is None:
                to_drop.append(key)
            elif not isinstance(val, (str, int, float, np.number, np.ndarray)):
                ds[var].attrs[key] = str(val)
        for key in to_drop:
            del ds[var].attrs[key]

    to_drop = []
    for key, val in ds.attrs.items():
        if val is None:
            to_drop.append(key)
        elif not isinstance(val, (str, int, float, np.number, np.ndarray)):
            ds.attrs[key] = str(val)
    for key in to_drop:
        del ds.attrs[key]

    return ds


# ─── Pre-Write Validation ──────────────────────────────────────────
import random

# ─── Pre-Write Validation ──────────────────────────────────────────

def pre_write_check(ds, year, n_samples=25, block_size=100):
    '''Monte Carlo spatial sampling to check for valid data.

    Randomly drops N blocks of 50x50 pixels across the grid.
    Prevents downloading 1.3 billion pixels just to verify integrity.
    '''
    data_vars = [v for v in ds.data_vars if v != 'spatial_ref']
    if not data_vars:
        return

    var_name = data_vars[0]
    da = ds[var_name]

    nx = da.sizes.get('x', 0)
    ny = da.sizes.get('y', 0)

    if nx == 0 or ny == 0:
        return

    # Adjust block size if grid is smaller than 50px
    bs = min(block_size, nx, ny)

    valid_found = False
    for i in range(n_samples):
        # Pick a random top-left corner
        rand_x = random.randint(0, nx - bs)
        rand_y = random.randint(0, ny - bs)

        indexers = {
            'x': slice(rand_x, rand_x + bs),
            'y': slice(rand_y, rand_y + bs)
        }
        if 'time' in da.dims:
            indexers['time'] = 0

        # Compute ONLY this tiny 50x50 block
        sample = da.isel(**indexers).compute()
        vals = sample.values

        if np.issubdtype(vals.dtype, np.floating):
            if np.isfinite(vals).sum() > 0:
                valid_found = True
                break
        else:
            fill = da.attrs.get('_FillValue', da.encoding.get('_FillValue', None))
            if fill is not None:
                if (vals != fill).sum() > 0:
                    valid_found = True
                    break
            else:
                valid_found = True
                break

    if not valid_found:
        raise DataQualityError(
            f'[{year}] Monte Carlo pre-check failed: No valid pixels found '
            f'in {n_samples} random samples. Check ROI/CRS.')

    log.info('[%d] Pre-check OK (Monte Carlo: %d random blocks sampled)', year, i + 1)


# ─── Post-Write Integrity Verification ─────────────────────────────

def post_write_verify(filepath, expected_vars, band_info, year):
    '''Reopen the NetCDF and verify structure + data integrity.

    FIX(11): checks ALL bands.
    FIX(16): now expects legitimate scale_factor on bands that declare it
             in BAND_METADATA (no longer flags them as "bogus").
    '''
    try:
        vds = xr.open_dataset(filepath, chunks='auto', mask_and_scale=False)
    except Exception as e:
        return False, f'Cannot reopen: {e}'

    try:
        missing = [v for v in expected_vars if v not in vds.data_vars]
        if missing:
            return False, f'Missing vars: {missing}'

        if 'time' not in vds.dims:
            return False, 'No time dimension'

        # grid_mapping present?
        if 'spatial_ref' not in vds:
            return False, 'Missing spatial_ref grid_mapping variable'

        # Coordinates finite?
        for c in ('x', 'y'):
            if c in vds.coords:
                vals = vds[c].values
                if np.issubdtype(vals.dtype, np.floating):
                    if not np.all(np.isfinite(vals)):
                        return False, f'Non-finite coord: {c}'

        # Check scale_factor: should match BAND_METADATA (not CRS pixel size)
        for v in expected_vars:
            info = band_info.get(v, {})
            expected_sf = info.get('scale_factor')
            actual_sf = vds[v].attrs.get('scale_factor',
                                         vds[v].encoding.get('scale_factor'))

            if expected_sf is not None:
                if actual_sf is None:
                    return False, f'{v}: expected scale_factor={expected_sf}, got none'
                if abs(float(actual_sf) - float(expected_sf)) > 1e-10:
                    return False, (
                        f'{v}: scale_factor={actual_sf} != expected {expected_sf} '
                        '(possible XEE pollution)')
            else:
                # Band with no expected scale — should have none
                if actual_sf is not None and float(actual_sf) != 1.0:
                    return False, (
                        f'{v}: unexpected scale_factor={actual_sf}')

        # Data has valid values? (sample ALL variables)
        bad_vars = []
        for v in expected_vars:
            da = vds[v]
            info = band_info.get(v, {})
            fill_val = info.get('_FillValue')

            idx = {}
            if 'time' in da.dims:
                idx['time'] = 0
            for d in da.dims:
                if d != 'time':
                    s = da.sizes[d]
                    m = s // 2
                    hw = max(1, min(50, s // 4))
                    idx[d] = slice(max(0, m - hw), min(s, m + hw))

            samp = da.isel(**idx).compute().values

            if np.issubdtype(samp.dtype, np.floating):
                valid = np.any(np.isfinite(samp))
            else:
                if fill_val is not None:
                    valid = np.any(samp != fill_val)
                else:
                    valid = samp.size > 0

            if not valid:
                bad_vars.append(v)

        if bad_vars:
            return False, f'No valid data on re-read: {bad_vars}'

        return True, f'All {len(expected_vars)} vars verified with correct metadata'

    finally:
        vds.close()


# ─── Drive Copy ─────────────────────────────────────────────────────

def get_md5(filepath, chunk_bytes=10 * 1024 * 1024):
    '''MD5 with standard buffered I/O (no O_DIRECT — FUSE incompatible).'''
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    '''Copy to Drive, flush, verify MD5.'''
    src_md5 = get_md5(src)

    for attempt in range(1, max_retries + 1):
        try:
            log.info('Drive copy attempt %d/%d ...', attempt, max_retries)

            with open(src, 'rb') as f_src, open(dst, 'wb') as f_dst:
                shutil.copyfileobj(f_src, f_dst, length=10 * 1024 * 1024)
                f_dst.flush()
                os.fsync(f_dst.fileno())

            os.sync()
            time.sleep(3)

            src_size = os.path.getsize(src)
            dst_size = os.path.getsize(dst)
            if src_size != dst_size:
                raise IOError(
                    f'Size mismatch: src={src_size}, dst={dst_size}')

            dst_md5 = get_md5(dst)
            if src_md5 != dst_md5:
                raise IOError('MD5 mismatch')

            log.info('Drive copy verified (MD5 match, %d bytes)', src_size)
            return True

        except Exception as e:
            log.error('Drive copy attempt %d failed: %s', attempt, e)
            if os.path.exists(dst):
                try:
                    os.remove(dst)
                except OSError:
                    pass
            if attempt < max_retries:
                wait = 15 * attempt
                log.info('Retrying in %ds ...', wait)
                time.sleep(wait)
            else:
                raise DataQualityError(
                    f'Drive copy failed after {max_retries} attempts: {e}')


# ╔════════════════════════════════════════════════════════════════════╗
# ║  MAIN PIPELINE                                                     ║
# ╚════════════════════════════════════════════════════════════════════╝

def run_pipeline():
    log.info('=' * 62)
    log.info('  Dataset : %s', DATASET_ID)
    log.info('=' * 62)

    # ── 1. Collection metadata ──────────────────────────────────
    coll = ee.ImageCollection(DATASET_ID)
    bands = coll.first().bandNames().getInfo()
    if not bands:
        raise ValueError(f'{DATASET_ID} returned no bands')

    try:
        y0 = YEAR_START or int(ee.Date(
            coll.sort('system:time_start').first()
                .get('system:time_start')).get('year').getInfo())
        y1 = YEAR_END or int(ee.Date(
            coll.sort('system:time_start', False).first()
                .get('system:time_start')).get('year').getInfo())
    except Exception as exc:
        raise ValueError(
            f'Cannot determine year range: {exc}') from exc

    years = list(range(y0, y1 + 1))
    log.info('Bands (%d) : %s', len(bands), bands)
    log.info('Years      : %d - %d  (%d total)', y0, y1, len(years))

    # ── 2. CRS & grid ──────────────────────────────────────────
    grid_crs, nominal_m = resolve_native_crs(coll.select(bands))

    tolerance_deg = (nominal_m * 1.5) / 111_320.0
    roi = load_roi(SHAPEFILE_DIR, simplify_deg=tolerance_deg)

    short = DATASET_ID.split('/')[-1]
    out_dir = os.path.join(DRIVE_ROOT, short)
    os.makedirs(out_dir, exist_ok=True)
    log.info('Output : %s', out_dir)

    grid_params = build_grid_params(roi, grid_crs, nominal_m)
    crs_transform = grid_params['crs_transform']
    grid_shape    = grid_params['shape_2d']

    # ── 3. Band metadata (auto-detect + user override) ──────────
    log.info('Detecting band types from GEE ...')
    auto_types = detect_band_types(coll.select(bands), bands)
    band_info  = merge_band_info(bands, BAND_METADATA, auto_types)

    log.info('Band info summary:')
    for b in bands:
        info = band_info[b]
        sf = info.get('scale_factor', '—')
        fv = info.get('_FillValue', '—')
        log.info('  %-20s  dtype=%-7s  scale=%-8s  fill=%s',
                 b, info['dtype'], sf, fv)

    # ── 4. Year loop ────────────────────────────────────────────
    n_ok = n_fail = n_skip = 0

    for year in years:
        fname = f'{short}_{year}_{ROI_LABEL}.nc'
        final = os.path.join(out_dir, fname)

        # ── Resume check ────────────────────────────────────────
        if os.path.exists(final) and not FORCE_OVERWRITE:
            sz = os.path.getsize(final)
            if sz < 1024:
                log.warning('[%d] Tiny file (%d B) — re-processing', year, sz)
                os.remove(final)
            else:
                is_valid = False
                try:
                    with xr.open_dataset(final, mask_and_scale=False) as chk:
                        if ('time' in chk.dims and chk.sizes['time'] > 0
                                and 'spatial_ref' in chk):
                            is_valid = True
                except Exception:
                    pass

                if is_valid:
                    log.info('[%d] On Drive (%.1f MiB), valid — skip',
                             year, sz / 1024**2)
                    n_skip += 1
                    continue
                else:
                    log.warning('[%d] Existing file missing metadata — re-processing', year)
                    try:
                        os.remove(final)
                    except OSError:
                        pass

        tmp = os.path.join('/content', fname)
        done = False

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                log.info('[%d] attempt %d/%d', year, attempt, MAX_RETRIES)


                # Mosaic the 49 images into 1, assign a dummy date,
                # and wrap back into an ImageCollection for XEE
                dummy_date = ee.Date.fromYMD(year, 1, 1)

                mosaic_img = (
                    ee.ImageCollection(DATASET_ID)
                    .select(bands)
                    .mosaic()
                    .set('system:time_start', dummy_date.millis())
                )

                yr_col = ee.ImageCollection([mosaic_img])

                n_img = yr_col.size().getInfo()
                if n_img == 0:
                    log.warning('[%d] 0 images — skip year', year)
                    n_skip += 1
                    break

                log.info('[%d] %d image(s) after mosaic', year, n_img)

                n_img = yr_col.size().getInfo()
                if n_img == 0:
                    log.warning('[%d] 0 images — skip year', year)
                    n_skip += 1
                    break

                log.info('[%d] %d image(s)', year, n_img)

                # ── Open lazily ──────────────────────────────────
                grid_w, grid_h = grid_shape
                safe_chunk = max(16, min(CHUNK_XY, grid_w, grid_h))

                ds = xr.open_dataset(
                    yr_col,
                    engine='ee',
                    chunks={'x': safe_chunk, 'y': safe_chunk},
                    **grid_params,
                )

                # Deduplicate timestamps
                _, unique_idx = np.unique(ds['time'], return_index=True)
                if len(unique_idx) != ds.sizes['time']:
                    log.warning('[%d] Deduplicating timestamps', year)
                    ds = ds.isel(time=sorted(unique_idx))

                # ── Fix encoding ─────────────────────────────────
                ds = strip_xee_encoding(ds)

                # ── Apply band metadata (scale, fill, units) ─────
                ds = apply_band_metadata(ds, band_info)

                # ── Cast to native dtypes (NaN → fill for ints) ──
                ds = prepare_for_write(ds, band_info)

                # ── CF compliance + grid_mapping ─────────────────
                ds = sanitize_attrs(ds)
                ds = make_cf_compliant(
                    ds, DATASET_ID, ROI_LABEL, grid_crs,
                    crs_transform, grid_shape, year)

                # ── Pre-write validation ─────────────────────────
                pre_write_check(ds, year)

                # ── Write to scratch ─────────────────────────────
                enc = build_encoding(ds, band_info, COMPRESS_LEVEL, safe_chunk)

                log.info('[%d] Writing -> %s', year, tmp)
                t0 = time.time()

                try:
                    with dask.config.set(scheduler='synchronous'):
                        ds.to_netcdf(
                            tmp,
                            engine='netcdf4',
                            encoding=enc,
                            unlimited_dims=['time'],
                        )
                except Exception:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                    raise

                dt = time.time() - t0
                mb = os.path.getsize(tmp) / 1024**2
                log.info('[%d] Written in %.0fs  (%.1f MiB)', year, dt, mb)

                if os.path.getsize(tmp) < 1024:
                    raise DataQualityError(
                        f'[{year}] File too small ({os.path.getsize(tmp)} B)')

                # ── Post-write integrity ─────────────────────────
                ok_flag, msg = post_write_verify(tmp, bands, band_info, year)
                if not ok_flag:
                    raise DataQualityError(
                        f'[{year}] INTEGRITY FAIL: {msg}')
                log.info('[%d] Integrity OK: %s', year, msg)

                # ── Export to Drive ───────────────────────────────
                log.info('[%d] Exporting to Drive ...', year)
                t_mv = time.time()
                robust_drive_copy(tmp, final, max_retries=3)
                os.remove(tmp)
                log.info('[%d] Exported in %.0fs -> %s',
                         year, time.time() - t_mv, final)

                n_ok += 1
                done = True
                break

            except DataQualityError as e:
                log.error('[%d] DATA ERROR (not retrying): %s', year, e)
                n_fail += 1
                break

            except Exception as e:
                log.error('[%d] attempt %d error: %s', year, attempt, e)
                if attempt < MAX_RETRIES:
                    wait = 30 * 2 ** (attempt - 1)
                    log.info('[%d] retry in %ds ...', year, wait)
                    time.sleep(wait)
                else:
                    log.error('[%d] retries exhausted', year)
                    n_fail += 1

            finally:

                if not done and os.path.exists(tmp):
                    try:
                        os.remove(tmp)
                    except OSError:
                        pass
                gc.collect()
                gc.collect()

    # ── Summary ─────────────────────────────────────────────────
    log.info('=' * 62)
    log.info('  DONE  %d ok | %d fail | %d skip  (%d years)',
             n_ok, n_fail, n_skip, len(years))
    log.info('=' * 62)

    if os.path.isdir(out_dir):
        nc_files = sorted(f for f in os.listdir(out_dir) if f.endswith('.nc'))
        if nc_files:
            log.info('Files on Drive (%s):', out_dir)
            for f in nc_files:
                sz = os.path.getsize(os.path.join(out_dir, f)) / 1024**2
                log.info('  %s  (%.1f MiB)', f, sz)


if __name__ == '__main__':
    run_pipeline()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.1 MB/s eta 0:00:00


# GRACE and GRACE FO
### NASA/GRACE/MASS_GRIDS_V04/MASCON

In [ ]:
# ==========================================
# 1. Install Required Packages & Setup Logging
# ==========================================
!pip install -q xee xarray netcdf4 geopandas geemap pyproj dask
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
gee_to_netcdf_v3.py — Earth Engine → CF-Compliant NetCDF Exporter
===================================================================

Changes from v2
---------------
FIX(16) scale_factor/add_offset written per-band as CF variable attributes
        so raw packed values can be decoded by any CF-compliant reader.
FIX(17) Per-band _FillValue: each variable gets its own native fill value.
        QA bitmask bands get uint16 dtype; angle bands get scale=0.01; etc.
FIX(18) grid_mapping variable (`spatial_ref`) added to every file.
        Contains full CRS WKT, GeoTransform, and CF parameters.
        Every data variable references it via grid_mapping='spatial_ref'.
FIX(19) Native integer dtypes preserved (int16 for NDVI, uint16 for QA).
        v2 cast everything to float32, losing packing + fill semantics.
FIX(20) Auto-detect band dtypes from GEE when BAND_METADATA is empty.
        Falls back to the detected dtype, logs warning about missing scale.

All v2 crash/logic/integrity fixes (FIX 1-15) are carried forward.
"""

# ╔════════════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION                                                     ║
# ╚════════════════════════════════════════════════════════════════════╝

DATASET_ID = "NASA/GRACE/MASS_GRIDS_V04/MASCON"
PROJECT_ID = "msugw-503806"

SHAPEFILE_DIR = "/content/ogallala_shp"
ROI_LABEL     = "Ogallala"
DRIVE_ROOT    = "/content/drive/MyDrive/MSUGWB"

YEAR_START = None
YEAR_END   = None

DASK_WORKERS    = 4
CHUNK_XY        = 512
COMPRESS_LEVEL  = 0
MAX_RETRIES     = 3
FORCE_OVERWRITE = False

# ── Per-Band Metadata (FIX 16/17/19) ────────────────────────────────
#
# Source: GEE Data Catalog for MODIS/061/MOD13A3
# https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13A3
#
# Keys per band:
#   dtype        – NetCDF storage type (int16, uint16, float32, …)
#   scale_factor – CF scale: physical = raw × scale + offset
#   add_offset   – CF offset (usually 0.0)
#   _FillValue   – raw value meaning "no data"
#   valid_range   – [min, max] of meaningful raw values (optional, CF attr)
#   long_name    – human-readable description (CF attr)
#   units        – physical units after decoding (CF attr)
#
# For a different dataset:
#   1. Change DATASET_ID
#   2. Replace BAND_METADATA with entries from the GEE catalog page
#   3. If you don't know the metadata, set BAND_METADATA = {}
#      → the pipeline will auto-detect dtypes and warn about missing scale
#

BAND_METADATA = {}


# ╔════════════════════════════════════════════════════════════════════╗
# ║  SETUP                                                             ║
# ╚════════════════════════════════════════════════════════════════════╝

# --- Colab prerequisites ---
# Uncomment the next two lines when running on Colab:
# !pip install -q "xee>=0.0.14" xarray netcdf4 geopandas pyproj
# from google.colab import drive; drive.mount('/content/drive')

import os, sys, glob, time, shutil, logging, gc, hashlib
from datetime import datetime, timezone

import numpy as np
import ee
import xarray as xr
import xee
from xee import helpers
import geopandas as gpd
import pyproj
import dask

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('gee_export')
logging.getLogger('urllib3.connectionpool').setLevel(logging.ERROR)

dask.config.set(scheduler='threads', num_workers=DASK_WORKERS)

ee.Authenticate()
ee.Initialize(project=PROJECT_ID, opt_url='https://earthengine-highvolume.googleapis.com')
log.info('Earth Engine ready  (project=%s)', PROJECT_ID)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  EXCEPTIONS                                                        ║
# ╚════════════════════════════════════════════════════════════════════╝

class DataQualityError(Exception):
    """Non-retryable data quality error."""
    pass


# ╔════════════════════════════════════════════════════════════════════╗
# ║  PIPELINE FUNCTIONS                                                ║
# ╚════════════════════════════════════════════════════════════════════╝

# ─── ROI ─────────────────────────────────────────────────────────────

def load_roi(shapefile_dir, simplify_deg=0.001):
    '''Load shapefile -> dissolved EPSG:4326 geometry.'''
    paths = glob.glob(os.path.join(shapefile_dir, '**', '*.shp'), recursive=True)
    if not paths:
        raise FileNotFoundError(f'No .shp found in {shapefile_dir}')
    log.info('Shapefile: %s', paths[0])

    gdf = gpd.read_file(paths[0])
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    roi = (gdf.geometry.union_all()
           if hasattr(gdf.geometry, 'union_all')
           else gdf.geometry.unary_union)

    try:
        import shapely as _shp
        nv = int(_shp.get_num_coordinates(roi))
    except Exception:
        nv = len(roi.wkt) // 20

    if nv > 50_000:
        roi = roi.simplify(simplify_deg, preserve_topology=True)
        try:
            nv_after = int(_shp.get_num_coordinates(roi))
        except Exception:
            nv_after = len(roi.wkt) // 20
        log.warning('Simplified ROI: %s -> %s vertices (tol=%.4f deg)',
                    f'{nv:,}', f'{nv_after:,}', simplify_deg)
    else:
        log.info('ROI vertices: %s', f'{nv:,}')
    return roi


# ─── CRS & Grid ─────────────────────────────────────────────────────

def resolve_native_crs(collection):
    '''Detect native CRS + nominal scale.  Returns (epsg_str, metres).'''
    proj = collection.first().select(0).projection()
    info = proj.getInfo()
    native_crs = info.get('crs', 'EPSG:4326')
    nominal_m  = proj.nominalScale().getInfo()

    log.info('Native CRS    : %s', native_crs[:70])
    log.info('Nominal scale : %.1f m', nominal_m)

    resolved = 'EPSG:4326'

    if native_crs.startswith('SR-ORG:'):
        log.warning('SR-ORG CRS (%s) -> EPSG:4326 fallback', native_crs)
    elif native_crs.startswith('EPSG:'):
        resolved = native_crs
    else:
        try:
            crs_obj = pyproj.CRS.from_user_input(native_crs)
            epsg = crs_obj.to_epsg()
            if epsg:
                resolved = f'EPSG:{epsg}'
            elif crs_obj.axis_info and crs_obj.axis_info[0].unit_name == 'metre':
                resolved = 'EPSG:5070'
                log.warning('Metre CRS, no EPSG -> EPSG:5070')
        except Exception as exc:
            log.warning('CRS parse error: %s -> EPSG:4326', exc)

    log.info('Resolved CRS  : %s', resolved)
    return resolved, nominal_m


def build_grid_params(roi, grid_crs, nominal_m):
    '''Compute crs_transform + shape_2d for the actual ROI subset.

    Returns the dict that XEE's open_dataset() expects.
    The crs_transform describes THIS subset, not the global grid (FIX 18).
    '''
    minx, miny, maxx, maxy = roi.bounds

    transformer = pyproj.Transformer.from_crs(
        'EPSG:4326', grid_crs, always_xy=True)
    x1, y1 = transformer.transform(minx, miny)
    x2, y2 = transformer.transform(maxx, maxy)

    left   = min(x1, x2)
    right  = max(x1, x2)
    bottom = min(y1, y2)
    top    = max(y1, y2)

    try:
        unit = pyproj.CRS.from_user_input(grid_crs).axis_info[0].unit_name
    except Exception:
        unit = 'degree'

    if unit == 'metre':
        scale_x = float(nominal_m)
        scale_y = float(nominal_m)
    else:
        lat_centroid = roi.centroid.y
        cos_lat = np.cos(np.radians(lat_centroid))
        scale_y = nominal_m / 111_320.0
        scale_x = scale_y / cos_lat if cos_lat > 0.01 else scale_y

    width  = int(np.ceil((right - left) / scale_x))
    height = int(np.ceil((top - bottom) / scale_y))

    crs_transform = (scale_x, 0.0, left, 0.0, -scale_y, top)

    log.info('Grid : %d x %d px  |  scale_x=%.6g  scale_y=%.6g %s',
             width, height, scale_x, scale_y, unit)

    return {
        'crs': grid_crs,
        'crs_transform': crs_transform,
        'shape_2d': (width, height),
    }


# ─── Band Metadata Detection (FIX 20) ───────────────────────────────

def detect_band_types(collection, bands):
    '''Auto-detect native dtypes from GEE ee.Image.bandTypes().

    Returns {band_name: {'dtype': 'int16'|'uint16'|'float32'|...}}.
    '''
    detected = {}
    try:
        img = collection.first()
        bt = img.bandTypes().getInfo()
        for band_name in bands:
            if band_name not in bt:
                continue
            info = bt[band_name]
            precision = info.get('precision', 'float')
            mn = info.get('min', 0)
            mx = info.get('max', 0)

            if precision == 'int':
                if mn >= 0 and mx <= 255:
                    dtype = 'uint8'
                elif mn >= 0 and mx <= 65535:
                    dtype = 'uint16'
                elif mn >= -32768 and mx <= 32767:
                    dtype = 'int16'
                else:
                    dtype = 'int32'
            elif precision == 'double':
                dtype = 'float64'
            else:
                dtype = 'float32'

            detected[band_name] = {'dtype': dtype}
            log.info('  Band %-20s : %s (GEE: %s, range [%s, %s])',
                     band_name, dtype, precision, mn, mx)
    except Exception as e:
        log.warning('Cannot auto-detect band types: %s', e)

    return detected


def merge_band_info(bands, user_metadata, auto_detected):
    '''Merge user-provided BAND_METADATA with auto-detected types.

    User metadata takes precedence.  Auto-detected provides dtype fallback.
    Bands with no metadata at all get float32 + warning.
    '''
    result = {}
    warned = False

    for band in bands:
        info = {}

        # Layer 1: auto-detected dtype
        if band in auto_detected:
            info.update(auto_detected[band])

        # Layer 2: user-provided overrides everything
        if band in user_metadata:
            info.update(user_metadata[band])

        # Ensure dtype exists
        if 'dtype' not in info:
            info['dtype'] = 'float32'

        # Default _FillValue based on dtype if not user-specified
        if '_FillValue' not in info:
            dt = np.dtype(info['dtype'])
            if np.issubdtype(dt, np.unsignedinteger):
                info['_FillValue'] = int(np.iinfo(dt).max)       # e.g. 65535
            elif np.issubdtype(dt, np.signedinteger):
                info['_FillValue'] = int(np.iinfo(dt).min)       # e.g. -32768
            else:
                info['_FillValue'] = None   # NaN for floats (handled by encoding)

        # Warn about missing scale_factor (only once)
        if 'scale_factor' not in info and not warned:
            if band not in user_metadata:
                log.warning(
                    'BAND_METADATA missing for "%s" (and possibly others). '
                    'Raw values will be written without scale_factor/add_offset. '
                    'Populate BAND_METADATA from the GEE catalog for proper CF decoding.',
                    band)
                warned = True

        result[band] = info

    return result


# ─── Encoding & Metadata ────────────────────────────────────────────

def strip_xee_encoding(ds):
    '''Clear ALL XEE encoding (the bogus CRS-derived scale_factor).

    GEE already returns raw pixel values via computePixels.  XEE injects
    the CRS pixel size as scale_factor, which is NOT a CF data-packing
    parameter.  Clearing it prevents xarray from mis-interpreting it.
    '''
    for name in list(ds.data_vars) + list(ds.coords):
        if name in ds:
            ds[name].encoding.clear()
    return ds


def apply_band_metadata(ds, band_info):
    '''Write scale_factor, add_offset, valid_range, units, long_name
    as VARIABLE ATTRIBUTES — not encoding.

    FIX(16): scale_factor/add_offset are CF variable attributes.
    xarray's encoding dict interprets them as packing instructions
    (it would divide data by scale_factor before writing).  Since our
    data is ALREADY packed as raw integers from GEE, we must NOT put
    scale/offset in the encoding.  Writing them as attrs means:
      - xarray writes the raw values as-is
      - scale_factor appears in the file as a variable attribute
      - any CF reader (xarray, CDO, ncview) decodes automatically

    FIX(17): Per-band _FillValue is set individually.
    '''
    for var in list(ds.data_vars):
        if var not in band_info:
            continue
        info = band_info[var]

        # CF scale/offset as attributes
        if 'scale_factor' in info:
            ds[var].attrs['scale_factor'] = np.float64(info['scale_factor'])
        if 'add_offset' in info:
            ds[var].attrs['add_offset'] = np.float64(info['add_offset'])

        # Optional CF attributes
        if 'valid_range' in info:
            ds[var].attrs['valid_range'] = np.array(
                info['valid_range'], dtype=info.get('dtype', 'float32'))
        if 'long_name' in info:
            ds[var].attrs['long_name'] = info['long_name']
        if 'units' in info:
            ds[var].attrs['units'] = info['units']
        if 'flag_meanings' in info:
            ds[var].attrs['flag_meanings'] = info['flag_meanings']

    return ds


def prepare_for_write(ds, band_info):
    '''Cast variables to their target dtype, filling NaN → _FillValue.

    FIX(19): preserves native integer dtypes instead of blanket float32.
    For int16/uint16 bands, NaN (a float concept) is replaced with the
    band's _FillValue before the cast to integer.

    All operations are lazy (dask) — no eager computation.
    '''
    for var in list(ds.data_vars):
        if var not in band_info:
            continue

        info = band_info[var]
        target_dtype = np.dtype(info['dtype'])
        fill_val = info.get('_FillValue')

        if np.issubdtype(target_dtype, np.integer):
            # Integer dtypes cannot represent NaN.
            # Replace NaN with the fill value, then cast.
            if fill_val is None:
                fill_val = int(np.iinfo(target_dtype).min)
            ds[var] = ds[var].fillna(fill_val).astype(target_dtype)
        else:
            # Float dtypes: NaN stays as NaN (natural missing indicator)
            ds[var] = ds[var].astype(target_dtype)

    return ds


def build_encoding(ds, band_info, compress_level=4, chunk_xy=512):
    '''Build NetCDF encoding dict with per-band dtype and _FillValue.

    FIX(3):  chunksizes capped to actual dims.
    FIX(17): per-band _FillValue.
    FIX(19): native dtype from band_info.

    IMPORTANT: scale_factor and add_offset are NOT in the encoding.
    They are variable attributes set by apply_band_metadata().
    '''
    enc = {}

    for var in ds.data_vars:
        info = band_info.get(var, {})
        dtype = info.get('dtype', 'float32')
        fill = info.get('_FillValue')

        # For float types with no explicit fill, use NaN
        if fill is None and np.issubdtype(np.dtype(dtype), np.floating):
            fill = np.float32(np.nan) if dtype == 'float32' else np.float64(np.nan)

        # Cast fill to the target dtype for HDF5 compatibility
        if fill is not None:
            try:
                fill = np.dtype(dtype).type(fill)
            except (ValueError, OverflowError):
                pass

        # Chunk sizes capped to actual dimensions
        var_chunks = []
        for dim in ds[var].dims:
            dim_size = ds[var].sizes[dim]
            if dim == 'time':
                var_chunks.append(min(1, dim_size))
            elif dim in ('x', 'y'):
                var_chunks.append(min(chunk_xy, dim_size))
            else:
                var_chunks.append(min(1, dim_size))

        enc[var] = {
            'dtype': dtype,
            '_FillValue': fill,
            'zlib': compress_level > 0,
            'complevel': compress_level,
            'chunksizes': tuple(var_chunks),
        }

    # Coordinate encoding
    for c in ds.coords:
        if c == 'time':
            enc[c] = {
                'dtype': 'int64',
                '_FillValue': None,
                'units': 'days since 1970-01-01',
                'calendar': 'proleptic_gregorian',
            }
        elif c in ('x', 'y'):
            enc[c] = {'dtype': 'float64', '_FillValue': None}

    return enc


# ─── CF Grid Mapping (FIX 18) ───────────────────────────────────────

def add_grid_mapping(ds, grid_crs, crs_transform, grid_shape):
    '''Add a CF-compliant grid_mapping variable (spatial_ref).

    FIX(18): The file previously had no grid_mapping despite declaring
    CF-1.8, and the CRS metadata described the global MODIS grid, not
    the actual ROI subset.

    This function creates a scalar `spatial_ref` variable with:
      - grid_mapping_name + all CF projection parameters (via pyproj)
      - Full CRS WKT for precision
      - GeoTransform describing THIS subset (not the global grid)
      - Actual grid dimensions
    Every data variable gets a grid_mapping='spatial_ref' attribute.
    '''
    crs_obj = pyproj.CRS.from_user_input(grid_crs)

    # pyproj.CRS.to_cf() returns all required CF parameters:
    #   grid_mapping_name, semi_major_axis, inverse_flattening, etc.
    cf_attrs = crs_obj.to_cf()

    # Add full WKT for tools that prefer it (GDAL, QGIS, rioxarray)
    wkt = crs_obj.to_wkt()
    cf_attrs['crs_wkt']     = wkt
    cf_attrs['spatial_ref'] = wkt   # GDAL convention

    # GeoTransform for THIS subset (GDAL convention):
    #   (x_origin, x_pixel_size, x_rotation, y_origin, y_rotation, y_pixel_size)
    # Our crs_transform: (x_scale, 0, x_origin, 0, -y_scale, y_origin)
    geo_transform = (
        f'{crs_transform[2]} {crs_transform[0]} {crs_transform[1]} '
        f'{crs_transform[5]} {crs_transform[3]} {crs_transform[4]}'
    )
    cf_attrs['GeoTransform'] = geo_transform

    # Grid dimensions of the actual subset
    width, height = grid_shape
    cf_attrs['grid_width']  = int(width)
    cf_attrs['grid_height'] = int(height)

    # Create the scalar grid_mapping variable
    ds['spatial_ref'] = xr.DataArray(
        data=np.int32(0),
        attrs=cf_attrs,
    )

    # Tag every data variable with grid_mapping
    for var in list(ds.data_vars):
        if var != 'spatial_ref':
            ds[var].attrs['grid_mapping'] = 'spatial_ref'

    return ds


def make_cf_compliant(ds, dataset_id, roi_label, grid_crs,
                      crs_transform, grid_shape, year):
    '''Stamp CF-1.8 global + coordinate attributes + grid_mapping.

    FIX(18): now includes grid_mapping variable and subset-specific
    GeoTransform instead of global grid metadata.
    '''
    width, height = grid_shape

    ds.attrs.update({
        'Conventions': 'CF-1.8',
        'title':  f'{dataset_id} -- {roi_label} ({year})',
        'source': f'Google Earth Engine: {dataset_id}',
        'history': f'Created {datetime.now(timezone.utc).isoformat()}',
        'crs': grid_crs,
        'geospatial_bounds_crs': grid_crs,
        'grid_width': int(width),
        'grid_height': int(height),
    })

    if 'time' in ds.coords:
        ds['time'].attrs.update(axis='T', standard_name='time')

    try:
        is_proj = (pyproj.CRS.from_user_input(grid_crs)
                   .axis_info[0].unit_name == 'metre')
    except Exception:
        is_proj = False

    if 'x' in ds.coords:
        ds['x'].attrs.update(
            axis='X',
            standard_name='projection_x_coordinate' if is_proj else 'longitude',
            units='m' if is_proj else 'degrees_east',
        )
    if 'y' in ds.coords:
        ds['y'].attrs.update(
            axis='Y',
            standard_name='projection_y_coordinate' if is_proj else 'latitude',
            units='m' if is_proj else 'degrees_north',
        )

    # Add grid_mapping variable
    ds = add_grid_mapping(ds, grid_crs, crs_transform, grid_shape)

    return ds


def sanitize_attrs(ds):
    '''Flatten complex GEE metadata; drop None values.'''
    for var in list(ds.data_vars) + list(ds.coords):
        if var not in ds:
            continue
        to_drop = []
        for key, val in ds[var].attrs.items():
            if val is None:
                to_drop.append(key)
            elif not isinstance(val, (str, int, float, np.number, np.ndarray)):
                ds[var].attrs[key] = str(val)
        for key in to_drop:
            del ds[var].attrs[key]

    to_drop = []
    for key, val in ds.attrs.items():
        if val is None:
            to_drop.append(key)
        elif not isinstance(val, (str, int, float, np.number, np.ndarray)):
            ds.attrs[key] = str(val)
    for key in to_drop:
        del ds.attrs[key]

    return ds


# ─── Pre-Write Validation ──────────────────────────────────────────

# def pre_write_check(ds, year):
#     '''Coarsened subsample across the entire grid, ALL variables.'''
#     for var_name in ds.data_vars:
#         if var_name == 'spatial_ref':
#             continue  # skip the grid_mapping scalar
#         da = ds[var_name]

#         coarsen_dims = {}
#         for d in ('x', 'y'):
#             if d in da.dims and da.sizes[d] > 1:
#                 coarsen_dims[d] = max(1, da.sizes[d] // 50)

#         if not coarsen_dims:
#             log.warning('[%d] %s: no spatial dims to check', year, var_name)
#             continue

#         indexers = {'time': 0} if 'time' in da.dims else {}

#         # Use .max() for int bands (no NaN propagation), check for
#         # the fill value instead of np.isfinite for integers
#         sample = (da.isel(**indexers)
#                     .coarsen(coarsen_dims, boundary='trim')
#                     .max()
#                     .compute())

#         vals = sample.values

#         if np.issubdtype(vals.dtype, np.floating):
#             nf = int(np.isfinite(vals).sum())
#         else:
#             # Integer: count non-fill values
#             fill = da.attrs.get('_FillValue',
#                                 da.encoding.get('_FillValue', None))
#             if fill is not None:
#                 nf = int((vals != fill).sum())
#             else:
#                 nf = int(vals.size)   # no fill → assume all valid

#         if nf == 0:
#             raise DataQualityError(
#                 f'[{year}] Variable "{var_name}" has no valid pixels.')
#         log.info('[%d] Pre-check %s: %d valid values OK', year, var_name, nf)

# ─── Pre-Write Validation ──────────────────────────────────────────

def pre_write_check(ds, year):
    '''Coarsened subsample across the entire grid, checking ONLY the first band.

    Checking all bands triggers duplicate network downloads from GEE.
    If the CRS/Geometry is broken, the first band will be NaN, which tells
    us everything we need to know without wasting time.
    '''
    # Find the first actual data variable (skip spatial_ref)
    data_vars = [v for v in ds.data_vars if v != 'spatial_ref']
    if not data_vars:
        return  # No data to check

    var_name = data_vars[0]
    da = ds[var_name]

    coarsen_dims = {}
    for d in ('x', 'y'):
        if d in da.dims and da.sizes[d] > 1:
            coarsen_dims[d] = max(1, da.sizes[d] // 50)

    if not coarsen_dims:
        log.warning('[%d] %s: no spatial dims to check', year, var_name)
        return

    indexers = {'time': 0} if 'time' in da.dims else {}

    # Use .max() for int bands (no NaN propagation), check for
    # the fill value instead of np.isfinite for integers
    sample = (da.isel(**indexers)
                .coarsen(coarsen_dims, boundary='trim')
                .max()
                .compute())

    vals = sample.values

    if np.issubdtype(vals.dtype, np.floating):
        nf = int(np.isfinite(vals).sum())
    else:
        # Integer: count non-fill values
        fill = da.attrs.get('_FillValue',
                            da.encoding.get('_FillValue', None))
        if fill is not None:
            nf = int((vals != fill).sum())
        else:
            nf = int(vals.size)   # no fill → assume all valid

    if nf == 0:
        raise DataQualityError(
            f'[{year}] Variable "{var_name}" has no valid pixels. '
            'Check GEE asset availability, ROI, and CRS.')

    log.info('[%d] Pre-check OK: %d valid values found in %s', year, nf, var_name)

# ─── Post-Write Integrity Verification ─────────────────────────────

def post_write_verify(filepath, expected_vars, band_info, year):
    '''Reopen the NetCDF and verify structure + data integrity.

    FIX(11): checks ALL bands.
    FIX(16): now expects legitimate scale_factor on bands that declare it
             in BAND_METADATA (no longer flags them as "bogus").
    '''
    try:
        vds = xr.open_dataset(filepath, chunks='auto', mask_and_scale=False)
    except Exception as e:
        return False, f'Cannot reopen: {e}'

    try:
        missing = [v for v in expected_vars if v not in vds.data_vars]
        if missing:
            return False, f'Missing vars: {missing}'

        if 'time' not in vds.dims:
            return False, 'No time dimension'

        # grid_mapping present?
        if 'spatial_ref' not in vds:
            return False, 'Missing spatial_ref grid_mapping variable'

        # Coordinates finite?
        for c in ('x', 'y'):
            if c in vds.coords:
                vals = vds[c].values
                if np.issubdtype(vals.dtype, np.floating):
                    if not np.all(np.isfinite(vals)):
                        return False, f'Non-finite coord: {c}'

        # Check scale_factor: should match BAND_METADATA (not CRS pixel size)
        for v in expected_vars:
            info = band_info.get(v, {})
            expected_sf = info.get('scale_factor')
            actual_sf = vds[v].attrs.get('scale_factor',
                                         vds[v].encoding.get('scale_factor'))

            if expected_sf is not None:
                if actual_sf is None:
                    return False, f'{v}: expected scale_factor={expected_sf}, got none'
                if abs(float(actual_sf) - float(expected_sf)) > 1e-10:
                    return False, (
                        f'{v}: scale_factor={actual_sf} != expected {expected_sf} '
                        '(possible XEE pollution)')
            else:
                # Band with no expected scale — should have none
                if actual_sf is not None and float(actual_sf) != 1.0:
                    return False, (
                        f'{v}: unexpected scale_factor={actual_sf}')

        # Data has valid values? (sample ALL variables)
        bad_vars = []
        for v in expected_vars:
            da = vds[v]
            info = band_info.get(v, {})
            fill_val = info.get('_FillValue')

            idx = {}
            if 'time' in da.dims:
                idx['time'] = 0
            for d in da.dims:
                if d != 'time':
                    s = da.sizes[d]
                    m = s // 2
                    hw = max(1, min(50, s // 4))
                    idx[d] = slice(max(0, m - hw), min(s, m + hw))

            samp = da.isel(**idx).compute().values

            if np.issubdtype(samp.dtype, np.floating):
                valid = np.any(np.isfinite(samp))
            else:
                if fill_val is not None:
                    valid = np.any(samp != fill_val)
                else:
                    valid = samp.size > 0

            if not valid:
                bad_vars.append(v)

        if bad_vars:
            return False, f'No valid data on re-read: {bad_vars}'

        return True, f'All {len(expected_vars)} vars verified with correct metadata'

    finally:
        vds.close()


# ─── Drive Copy ─────────────────────────────────────────────────────

def get_md5(filepath, chunk_bytes=10 * 1024 * 1024):
    '''MD5 with standard buffered I/O (no O_DIRECT — FUSE incompatible).'''
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    '''Copy to Drive, flush, verify MD5.'''
    src_md5 = get_md5(src)

    for attempt in range(1, max_retries + 1):
        try:
            log.info('Drive copy attempt %d/%d ...', attempt, max_retries)

            with open(src, 'rb') as f_src, open(dst, 'wb') as f_dst:
                shutil.copyfileobj(f_src, f_dst, length=10 * 1024 * 1024)
                f_dst.flush()
                os.fsync(f_dst.fileno())

            os.sync()
            time.sleep(3)

            src_size = os.path.getsize(src)
            dst_size = os.path.getsize(dst)
            if src_size != dst_size:
                raise IOError(
                    f'Size mismatch: src={src_size}, dst={dst_size}')

            dst_md5 = get_md5(dst)
            if src_md5 != dst_md5:
                raise IOError('MD5 mismatch')

            log.info('Drive copy verified (MD5 match, %d bytes)', src_size)
            return True

        except Exception as e:
            log.error('Drive copy attempt %d failed: %s', attempt, e)
            if os.path.exists(dst):
                try:
                    os.remove(dst)
                except OSError:
                    pass
            if attempt < max_retries:
                wait = 15 * attempt
                log.info('Retrying in %ds ...', wait)
                time.sleep(wait)
            else:
                raise DataQualityError(
                    f'Drive copy failed after {max_retries} attempts: {e}')


# ╔════════════════════════════════════════════════════════════════════╗
# ║  MAIN PIPELINE                                                     ║
# ╚════════════════════════════════════════════════════════════════════╝

def run_pipeline():
    log.info('=' * 62)
    log.info('  Dataset : %s', DATASET_ID)
    log.info('=' * 62)

    # ── 1. Collection metadata ──────────────────────────────────
    coll = ee.ImageCollection(DATASET_ID)
    bands = coll.first().bandNames().getInfo()
    if not bands:
        raise ValueError(f'{DATASET_ID} returned no bands')

    try:
        y0 = YEAR_START or int(ee.Date(
            coll.sort('system:time_start').first()
                .get('system:time_start')).get('year').getInfo())
        y1 = YEAR_END or int(ee.Date(
            coll.sort('system:time_start', False).first()
                .get('system:time_start')).get('year').getInfo())
    except Exception as exc:
        raise ValueError(
            f'Cannot determine year range: {exc}') from exc

    years = list(range(y0, y1 + 1))
    log.info('Bands (%d) : %s', len(bands), bands)
    log.info('Years      : %d - %d  (%d total)', y0, y1, len(years))

    # ── 2. CRS & grid ──────────────────────────────────────────
    grid_crs, nominal_m = resolve_native_crs(coll.select(bands))

    tolerance_deg = (nominal_m * 1.5) / 111_320.0
    roi = load_roi(SHAPEFILE_DIR, simplify_deg=tolerance_deg)

    short = "GRACE"
    out_dir = os.path.join(DRIVE_ROOT, short)
    os.makedirs(out_dir, exist_ok=True)
    log.info('Output : %s', out_dir)

    grid_params = build_grid_params(roi, grid_crs, nominal_m)
    crs_transform = grid_params['crs_transform']
    grid_shape    = grid_params['shape_2d']

    # ── 3. Band metadata (auto-detect + user override) ──────────
    log.info('Detecting band types from GEE ...')
    auto_types = detect_band_types(coll.select(bands), bands)
    band_info  = merge_band_info(bands, BAND_METADATA, auto_types)

    log.info('Band info summary:')
    for b in bands:
        info = band_info[b]
        sf = info.get('scale_factor', '—')
        fv = info.get('_FillValue', '—')
        log.info('  %-20s  dtype=%-7s  scale=%-8s  fill=%s',
                 b, info['dtype'], sf, fv)

    # ── 4. Year loop ────────────────────────────────────────────
    n_ok = n_fail = n_skip = 0

    for year in years:
        fname = f'{short}_{year}_{ROI_LABEL}.nc'
        final = os.path.join(out_dir, fname)

        # ── Resume check ────────────────────────────────────────
        if os.path.exists(final) and not FORCE_OVERWRITE:
            sz = os.path.getsize(final)
            if sz < 1024:
                log.warning('[%d] Tiny file (%d B) — re-processing', year, sz)
                os.remove(final)
            else:
                is_valid = False
                try:
                    with xr.open_dataset(final, mask_and_scale=False) as chk:
                        if ('time' in chk.dims and chk.sizes['time'] > 0
                                and 'spatial_ref' in chk):
                            is_valid = True
                except Exception:
                    pass

                if is_valid:
                    log.info('[%d] On Drive (%.1f MiB), valid — skip',
                             year, sz / 1024**2)
                    n_skip += 1
                    continue
                else:
                    log.warning('[%d] Existing file missing metadata — re-processing', year)
                    try:
                        os.remove(final)
                    except OSError:
                        pass

        tmp = os.path.join('/content', fname)
        done = False

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                log.info('[%d] attempt %d/%d', year, attempt, MAX_RETRIES)

                yr_col = (
                    ee.ImageCollection(DATASET_ID)
                    .filter(ee.Filter.calendarRange(year, year, 'year'))
                    .select(bands)
                    .sort('system:time_start')
                )

                n_img = yr_col.size().getInfo()
                if n_img == 0:
                    log.warning('[%d] 0 images — skip year', year)
                    n_skip += 1
                    break

                log.info('[%d] %d image(s)', year, n_img)

                # ── Open lazily ──────────────────────────────────
                grid_w, grid_h = grid_shape
                safe_chunk = max(16, min(CHUNK_XY, grid_w, grid_h))

                ds = xr.open_dataset(
                    yr_col,
                    engine='ee',
                    chunks={'x': safe_chunk, 'y': safe_chunk},
                    **grid_params,
                )

                # Deduplicate timestamps
                _, unique_idx = np.unique(ds['time'], return_index=True)
                if len(unique_idx) != ds.sizes['time']:
                    log.warning('[%d] Deduplicating timestamps', year)
                    ds = ds.isel(time=sorted(unique_idx))

                # ── Fix encoding ─────────────────────────────────
                ds = strip_xee_encoding(ds)

                # ── Apply band metadata (scale, fill, units) ─────
                ds = apply_band_metadata(ds, band_info)

                # ── Cast to native dtypes (NaN → fill for ints) ──
                ds = prepare_for_write(ds, band_info)

                # ── CF compliance + grid_mapping ─────────────────
                ds = sanitize_attrs(ds)
                ds = make_cf_compliant(
                    ds, DATASET_ID, ROI_LABEL, grid_crs,
                    crs_transform, grid_shape, year)

                # ── Pre-write validation ─────────────────────────
                pre_write_check(ds, year)

                # ── Write to scratch ─────────────────────────────
                enc = build_encoding(ds, band_info, COMPRESS_LEVEL, safe_chunk)

                log.info('[%d] Writing -> %s', year, tmp)
                t0 = time.time()

                try:
                    with dask.config.set(scheduler='synchronous'):
                        ds.to_netcdf(
                            tmp,
                            engine='netcdf4',
                            encoding=enc,
                            unlimited_dims=['time'],
                        )
                except Exception:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                    raise

                dt = time.time() - t0
                mb = os.path.getsize(tmp) / 1024**2
                log.info('[%d] Written in %.0fs  (%.1f MiB)', year, dt, mb)

                if os.path.getsize(tmp) < 1024:
                    raise DataQualityError(
                        f'[{year}] File too small ({os.path.getsize(tmp)} B)')

                # ── Post-write integrity ─────────────────────────
                ok_flag, msg = post_write_verify(tmp, bands, band_info, year)
                if not ok_flag:
                    raise DataQualityError(
                        f'[{year}] INTEGRITY FAIL: {msg}')
                log.info('[%d] Integrity OK: %s', year, msg)

                # ── Export to Drive ───────────────────────────────
                log.info('[%d] Exporting to Drive ...', year)
                t_mv = time.time()
                robust_drive_copy(tmp, final, max_retries=3)
                os.remove(tmp)
                log.info('[%d] Exported in %.0fs -> %s',
                         year, time.time() - t_mv, final)

                n_ok += 1
                done = True
                break

            except DataQualityError as e:
                log.error('[%d] DATA ERROR (not retrying): %s', year, e)
                n_fail += 1
                break

            except Exception as e:
                log.error('[%d] attempt %d error: %s', year, attempt, e)
                if attempt < MAX_RETRIES:
                    wait = 30 * 2 ** (attempt - 1)
                    log.info('[%d] retry in %ds ...', year, wait)
                    time.sleep(wait)
                else:
                    log.error('[%d] retries exhausted', year)
                    n_fail += 1

            finally:
                if not done and os.path.exists(tmp):
                    try:
                        os.remove(tmp)
                    except OSError:
                        pass
                gc.collect()
                gc.collect()

    # ── Summary ─────────────────────────────────────────────────
    log.info('=' * 62)
    log.info('  DONE  %d ok | %d fail | %d skip  (%d years)',
             n_ok, n_fail, n_skip, len(years))
    log.info('=' * 62)

    if os.path.isdir(out_dir):
        nc_files = sorted(f for f in os.listdir(out_dir) if f.endswith('.nc'))
        if nc_files:
            log.info('Files on Drive (%s):', out_dir)
            for f in nc_files:
                sz = os.path.getsize(os.path.join(out_dir, f)) / 1024**2
                log.info('  %s  (%.1f MiB)', f, sz)


if __name__ == '__main__':
    run_pipeline()



/tmp/ipykernel_7052/1302839578.py:976: UserWarning: The specified chunks separate the stored chunks along dimension "y" starting at index 16. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_7052/1302839578.py:976: UserWarning: The specified chunks separate the stored chunks along dimension "y" starting at index 16. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_7052/1302839578.py:976: UserWarning: The specified chunks separate the stored chunks along dimension "y" starting at index 16. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_7052/1302839578.py:976: UserWarning: The specified chunks separate the stored chunks along dimension "y" starting at index 16. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_7052/1302839578.p